# Epigraph Rank Scales + 889-Qubit DQI

Can an explicit L1 model take a useful step where smooth SQP stops? This small controlled study starts from [our scored Free-Gap SQP V1](https://www.kaggle.com/code/avazaku/free-gap-sqp-889-qubit-dqi?scriptVersionId=363660552), real and native Public Score **22.88871**. On 30 October 2026 at 03:21 UTC, Code > All > Public Score (including all data sources) displayed it second, tied at 22.88871 with Heliosli V22; Alex Chilton was third at 22.88856. That dated public-notebook comparison is not the overall leaderboard.

Our new effect is only **0.0338 ppm additional chemistry normalization saving**, about **0.00000338%**, beyond both the scored source and a same-start rank8 SQP control. This may leave the five-decimal competition display unchanged. The previous Free-Gap study's 19.921 ppm gain is prior work, not a new result here.

All **17 circuits**, architecture and encoded vectors are credited upstream work. We change positive rank amplitudes and compensate one-body offsets. The own contribution is a reusable L1 epigraph trust-region implementation, controlled negative screens, exact quadratic error model, and tiny independently checked feasible improvement. Standard linear programming and epigraph reformulation are not new algorithms.

This CPU run loads frozen endpoints, repeats no coefficient search, and creates a ZIP only after **10 NEW full official checks on two salts**. No new competition score is claimed by a numerical estimate.


## 1. Positive rank amplitudes with one-body compensation

Write $p_r=h_r-\sum_bW_{rb}$ and use positive scales $t_r$. Set $W'_r=t_rW_r$, $X'_r=t_rX_r$, $p'_r=p_r/t_r$, and $h'_r=p'_r+\sum_bW'_{rb}$. Then $\sum_r p'_rX'_r=\sum_rp_rX_r$, so the one-body target and error1 are preserved to floating precision. **The body Gram, error2 and gap change; the whole Hamiltonian is not invariant.**

With $u=t^2$, squared error2 is the exact quadratic $e_{20}+2(u-1)^Td+(u-1)^TA(u-1)$. The gap is $g_0+\frac12\sum_r\|p_r\|^2(1/u_r-1)$. We minimize the actual $\sqrt{g(2\Lambda-g)}$ with $g\ge3.9148981$ and error2 below 0.0199995. An LP retains all 480 absolute-value epigraph variables rather than differentiating through a fixed L1 face. Its candidate is bounded by the exact error quadratic and accepted only after evaluating the nonlinear objective and constraints. No stationarity or global-optimality proof follows.


In [ ]:
import os
os.environ.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2')
from pathlib import Path,PurePosixPath
import warnings,io,json,hashlib,base64,zipfile,tempfile,time,inspect,sys,gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,FileLink
from scipy.optimize import linear_sum_assignment,minimize,Bounds,linprog
from threadpoolctl import threadpool_limits
threadpool_limits(2)



COMPETITION_FILES={'kickmix/LICENSE.txt': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'kickmix/kickmix.so': 'c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e', 'verifiers/LICENSE.txt': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'verifiers/dfthcsos/FeMoCo76o113e.fcidump': 'e5a48af746fd075026dec9e8f5df2d363802a121e8c53af9a0b984bde695e1bd', 'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz': '7b7facf63fe35a9d9d3d00787ffdb700c863fa845fe4724d7f7c54778151cd87', 'verifiers/dfthcsos/SCORING_README.md': 'c2228569f96e58e118419bc32be2be9e33d6abffb8117cb0b91e1fc25b2a0230', 'verifiers/dfthcsos/__init__.py': '81915e324cc0cade67f52a54b7a51991fd96abaea94d1f8a4d423dae20e57348', 'verifiers/dfthcsos/scoreinfra/__init__.py': 'be90e0d503d5c20dcac0102af638b9a4ccad0341daa5baa5d49216a259e4516f', 'verifiers/dfthcsos/scoreinfra/circuits.py': 'fc027b52fdcebe14404f403a089122485d097ed65d03e6d989623d620b1ab68c', 'verifiers/dfthcsos/scoreinfra/costs_prepare.py': '7f1e190c42fc5325f3e6689fbbef51f95b0cb12f1c828cda92b6c088e72206da', 'verifiers/dfthcsos/scoreinfra/dfthc_math.py': 'ae82898d47a7a86b96378fb906e0f8a9f46003065cf19c7c09952a123b5b5af8', 'verifiers/dfthcsos/scoreinfra/reference.py': 'a2d5a63191d655729e29f0fde1e85fe4630bf896a91a63bb894d8d7daf71ed39', 'verifiers/dfthcsos/verify.py': '0775be7a528922367caebe9e4d051d6199811f1aefeaa4b793eaaa4621b8ec9d', 'verifiers/dqi/__init__.py': '01c870aa9763f4e4357fc6654bff1d3296cec995bd0943f87b4a48946ada5b40', 'verifiers/dqi/dqi_reference_circuit.py': '2b8e1b80a806157c333a7e308eaf2499ee09de499efbc05875202c327959959e', 'verifiers/dqi/verify_submission_dqi.py': 'f14274bc0943fb9769ddb01131be6063191a15009fe1caf1d4bdd9fd25194e65', 'verifiers/invsqrt/__init__.py': 'e9a6ecf253c343528578be5fa92eb820ccade63333aebb784552fb4ae75dbc0b', 'verifiers/invsqrt/nr_reference_circuit.py': 'bb90330dcf774f7b502de471293facba67f27952cc8808fe4a9970a9b1322e23', 'verifiers/invsqrt/verify_submission_invsqrt.py': 'ddef520dec6b68d1cf0a6f328ddd36d79eb84570b9192cf3f7acaf073f06733a'}
SOURCE_MEMBERS={'dfthcsos/angle_k_0.kmb': '9c31da0c1d047e7dcac638df807d3cef96eac6f736a8838a0ae6943e9c46c74f', 'dfthcsos/angle_k_1.kmb': '39b8d10098be097f75148639ee302249bdabe112a9278a48565eadb0cb86810b', 'dfthcsos/angle_k_10.kmb': 'e75f6b0685d754bf76f3f4d62589bf1d07b118faeeb0ab8745833733286db2fc', 'dfthcsos/angle_k_11.kmb': '773e54f6f0d7c3f769db433f263d541cdcff1076fb75109740907e3249996141', 'dfthcsos/angle_k_12.kmb': 'd86ca7ddd984059ded765b953cc936388517a7ca5789676ae90910a2be89769d', 'dfthcsos/angle_k_13.kmb': '50d82cf65116f06c0fe469ebba312f72675b277263461ab88b4d15eca947c050', 'dfthcsos/angle_k_14.kmb': 'ae2070998e31b7e05d01183914df20b6974c1760bddf90d1adac86ea75c7bf3f', 'dfthcsos/angle_k_2.kmb': '2e8dfdeb69c496e9e510fe99722153663760980d45c35ad9b72e79ba799dde49', 'dfthcsos/angle_k_3.kmb': '13817cf8d36ea112e62d7863edde2193aa5907dd0605612c4f0e1e54bacc173b', 'dfthcsos/angle_k_4.kmb': '8dce8f4a380998d818bf8a08e4f3daa62aaeb31a25d084aa57549f931a937f3d', 'dfthcsos/angle_k_5.kmb': '42cf7d76ef0f0b36640a9e65005214fdd4d9a9bb82b8cef6e658dbbdd8417099', 'dfthcsos/angle_k_6.kmb': '7fd7bd333b1427ecf546b0c27717ed9428e255e5165c4db14e5f95f8af45c838', 'dfthcsos/angle_k_7.kmb': '809a775b98836ee00e8783d3f616bb8517bb3a3621978bc704d20b543a8df4ec', 'dfthcsos/angle_k_8.kmb': '42d0deb376981d0470bebdd370ca2adf79685b4bf3e6909be262e2beced030ec', 'dfthcsos/angle_k_9.kmb': 'f1c49e7cf8e4f3a9adaffe167d2bbe21497a52eca2c7d09f765fc5a125e304c3', 'dfthcsos/submission.npz': 'dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6', 'dqi.kmb': '7a378f4755605e0c5fa6117f808891bf8dcf0797d047155e6d4c9e75e3c99421', 'invsqrt.kmb': '9a8627c812833a6feca6eb82239f38d1a9a5e1eee299a9a7af9abd85e36339be'}
ARCHIVE_SHA='196eaa849e3a28319d7df5bc1308993d23e9adfd35babdec4d4e8c1d47ea5335'
FACTOR_SHA='dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6'


## 2. Reusable helpers and independent controls

The spectral offset model is included for an independent parity check. `RankAmplitudeModel`, `rank_epigraph_step` and `refit_rank_amplitudes` are the reusable rank-scale implementation. The frozen runner never calls the search routine. It recomputes the exact error quadratic, tests two independent nonzero full-precision fixtures, three derivatives on each fixture, and a known LP epigraph solution. Two CPU threads are explicit.


In [ ]:
"""avazaku, Apache-2.0. Fixed-two-body, iso-gap identity-offset search.

The public XOR2 factor/circuits remain upstream work. This module varies
identity offsets while keeping the two-body weights fixed, and uses the unused
one-body residual margin. It is a numerical constrained experiment, not an exact
gauge transformation of the whole Hamiltonian or a global optimality claim.
"""
import time
import numpy as np


class OffsetModel:
    def __init__(self, weights, x, one_body_rows, base_h1, constant_gap, assignment=None):
        self.w = np.asarray(weights, dtype=np.float64).copy()
        self.shape = self.w.shape[::2]
        self.x = np.asarray(x, dtype=np.float64).reshape(-1, x.shape[-1], x.shape[-1])
        self.flat = self.x.reshape(len(self.x), -1)
        self.rows = np.asarray(one_body_rows, dtype=np.float64)
        self.base = np.asarray(base_h1, dtype=np.float64)
        self.gap_constant = float(constant_gap)
        self.sumw = self.w.sum(axis=1).ravel()
        self.l1w = np.abs(self.w).sum(axis=1).ravel()
        self.assignment = assignment

    def evaluate(self, offset, gradients=True):
        p = np.asarray(offset, dtype=np.float64).ravel()
        h = p + self.sumw
        matrix = self.base - (p @ self.flat).reshape(self.base.shape)
        eigenvalues, vectors = np.linalg.eigh(matrix)
        overlap = np.abs(self.rows @ vectors)
        if self.assignment is None:
            # Distinct row maxima certify the exact assignment optimum.
            matched = np.argmax(overlap, axis=1)
            if len(set(matched)) != len(matched):
                raise ValueError('Use scipy.optimize.linear_sum_assignment for ambiguous matching.')
        else:
            ii, jj = self.assignment(-overlap)
            matched = np.empty(len(eigenvalues), dtype=np.int64)
            matched[ii] = jj
        decoded = self.rows.T @ (eigenvalues[matched, None] * self.rows)
        residual = decoded - matrix
        e1sq = float(np.sum(residual*residual))
        lam1 = float(np.abs(eigenvalues).sum())
        lam2 = float(.25*np.sum((self.l1w + np.abs(h))**2))
        gap = float(self.gap_constant + lam1 - eigenvalues.sum() + .5*np.dot(p,p))
        result = dict(error_1=float(np.sqrt(e1sq)), error_1_sq=e1sq,
                      Lambda=lam1+lam2, E_gap=gap,
                      lambda_eff=float(np.sqrt(gap*(2*(lam1+lam2)-gap))))
        if gradients:
            sign_matrix = (vectors * np.sign(eigenvalues)) @ vectors.T
            grad_lam1 = -self.flat @ sign_matrix.ravel()
            row_coefficients = np.einsum('ij,jk,ik->i', self.rows, residual, self.rows, optimize=True)
            eigen_coefficients = np.empty_like(row_coefficients)
            eigen_coefficients[matched] = row_coefficients
            spectral_adjoint = (vectors * eigen_coefficients) @ vectors.T
            result.update(gradient_lambda=grad_lam1+.5*(self.l1w+np.abs(h))*np.sign(h),
                          gradient_lambda_smooth=grad_lam1+.5*h,
                          gradient_gap=grad_lam1+h,
                          gradient_error1_sq=2*self.flat @ (residual-spectral_adjoint).ravel())
        return result


def scalar_metrics(result):
    return {k:float(v) for k,v in result.items() if not k.startswith('gradient')}


def tangent_direction(result, active_accuracy):
    gradients=[result['gradient_gap']]
    if active_accuracy:
        gradients.append(result['gradient_error1_sq'])
    constraints=np.stack(gradients)
    g=result['gradient_lambda']
    # SVD pseudoinverse handles a redundant local constraint without regularizing
    # the physical target or accepting an energy violation.
    direction=-g+constraints.T @ (np.linalg.pinv(constraints @ constraints.T) @ (constraints @ g))
    norm=float(np.linalg.norm(direction))
    return direction/max(norm,1e-30)


def retract(model, point, gap_target, error_cap, active_accuracy, support_only=False):
    p=point.copy()
    for _ in range(10):
        try:
            r=model.evaluate(p)
        except ValueError as exc:
            if 'ambiguous matching' in str(exc):
                return None,None
            raise
        gap_defect=r['E_gap']-gap_target
        error_defect=r['error_1_sq']-error_cap**2
        if abs(gap_defect)<2e-10 and (not active_accuracy or abs(error_defect)<1e-12):
            return p,r
        gradients=[r['gradient_gap']]
        defects=[gap_defect]
        if active_accuracy:
            gradients.append(r['gradient_error1_sq'])
            defects.append(error_defect)
        jac=np.stack(gradients)
        if support_only:
            jac=jac*(np.abs(p+model.sumw)>1e-10)[None,:]
        correction=jac.T @ (np.linalg.pinv(jac @ jac.T) @ np.array(defects))
        if not np.isfinite(correction).all() or np.linalg.norm(correction)>10:
            break
        p-=correction
    return None,None


def optimize_offsets(model, initial_offset, seconds=120, callback=None, error_cap=.0199995):
    start=time.monotonic()
    p=np.asarray(initial_offset,dtype=np.float64).ravel().copy()
    current=model.evaluate(p)
    target=current['E_gap']
    assert current['error_1']<error_cap and target>3.914898
    initial=current['Lambda']
    history=[]
    accepted=trials=0
    step=.02
    last=-100.
    for iteration in range(5000):
        if time.monotonic()-start>=seconds:
            break
        active=current['error_1']>error_cap-2e-6
        direction=tangent_direction(current,active)
        moved=False
        for backtrack in range(20):
            if time.monotonic()-start>=seconds:
                break
            proposed,metrics=retract(model,p+step*direction,target,error_cap,active)
            trials+=1
            if (proposed is not None and metrics['error_1']<=error_cap+1e-10
                    and abs(metrics['E_gap']-target)<5e-10
                    and metrics['Lambda']<current['Lambda']-1e-10):
                p,current=proposed,metrics
                accepted+=1
                step=min(step*1.3,.1)
                moved=True
                break
            step*=.5
        elapsed=time.monotonic()-start
        if elapsed-last>=1 or not moved:
            row=dict(seconds=elapsed,iteration=iteration,accepted=accepted,trials=trials,
                     gap_target=target,active_accuracy=active,step=step,**scalar_metrics(current))
            history.append(row)
            if callback:
                callback(row,p.reshape(model.shape))
            last=elapsed
        if not moved or step<1e-12:
            break
    return p.reshape(model.shape),dict(initial_lambda=initial,final=scalar_metrics(current),
                                      gap_target=target,accepted=accepted,trials=trials,
                                      seconds=time.monotonic()-start,history=history)


def soft_threshold(x, threshold):
    return np.sign(x)*np.maximum(np.abs(x)-threshold,0.)


def proximal_tangent_step(model, p, result, step, active_accuracy):
    """Solve the convex local weighted-L1 step on the linearized constraints.

    One constraint uses its monotone scalar dual. Two use Dykstra's proximal
    composition with a tiny affine projection. This changes the treatment of
    L1 corners; the actual nonlinear objective and constraints still gate it.
    """
    h=p+model.sumw
    z=h-step*result['gradient_lambda_smooth']
    threshold=step*.5*model.l1w
    rows=[result['gradient_gap']]
    if active_accuracy:
        rows.append(result['gradient_error1_sq'])
    c=np.stack(rows)
    c/=np.maximum(np.linalg.norm(c,axis=1,keepdims=True),1e-30)
    if len(c)==1:
        direction=c[0]
        def trial(mu):
            x=soft_threshold(z-mu*direction,threshold)
            return float(direction @ (x-h)),x
        lo,hi=-1.,1.
        for _ in range(40):
            if trial(lo)[0]>=0 and trial(hi)[0]<=0:
                break
            lo*=2;hi*=2
        for _ in range(65):
            mid=.5*(lo+hi)
            value,x=trial(mid)
            if abs(value)<1e-13:
                break
            if value>0:
                lo=mid
            else:
                hi=mid
    else:
        inv=np.linalg.pinv(c @ c.T)
        x=z.copy();q1=np.zeros_like(x);q2=q1.copy()
        for _ in range(500):
            y=soft_threshold(x+q1,threshold);q1=x+q1-y
            xx=y+q2
            new=xx-c.T @ (inv @ (c @ (xx-h)))
            q2=xx-new
            if np.max(np.abs(new-x))<1e-12:
                x=new
                break
            x=new
        # Reapply soft threshold through the converged dual, keeping exact zeros.
        dual_projection=q2
        x=soft_threshold(z-dual_projection,threshold)
    if np.max(np.abs(c @ (x-h)))>1e-8:
        return None
    return x-model.sumw


def optimize_prox_offsets(model, initial_offset, seconds=120, callback=None, error_cap=.0199995, target_gap=None):
    start=time.monotonic()
    p=np.asarray(initial_offset,dtype=np.float64).ravel().copy()
    current=model.evaluate(p);target=current['E_gap'] if target_gap is None else float(target_gap);initial=current['Lambda']
    assert current['error_1']<=error_cap+1e-10 and abs(current['E_gap']-target)<1e-8 and target>3.914898
    history=[];accepted=trials=0;step=.01;last=-100.
    for iteration in range(5000):
        if time.monotonic()-start>=seconds:
            break
        active=current['error_1']>error_cap-2e-6
        moved=False
        for _ in range(25):
            if time.monotonic()-start>=seconds:
                break
            point=proximal_tangent_step(model,p,current,step,active)
            proposed,metrics=(None,None) if point is None else retract(model,point,target,error_cap,active,support_only=True)
            trials+=1
            if (proposed is not None and metrics['error_1']<=error_cap+1e-10
                    and abs(metrics['E_gap']-target)<5e-10
                    and metrics['Lambda']<current['Lambda']-1e-10):
                p,current=proposed,metrics;accepted+=1;step=min(step*1.4,.5);moved=True
                break
            step*=.5
        elapsed=time.monotonic()-start
        if elapsed-last>=1 or not moved:
            row=dict(method='proximal',seconds=elapsed,iteration=iteration,accepted=accepted,trials=trials,
                     gap_target=target,active_accuracy=active,step=step,**scalar_metrics(current))
            history.append(row)
            if callback:
                callback(row,p.reshape(model.shape))
            last=elapsed
        if not moved or step<1e-12:
            break
    return p.reshape(model.shape),dict(method='proximal',initial_lambda=initial,final=scalar_metrics(current),
                                      gap_target=target,accepted=accepted,trials=trials,
                                      seconds=time.monotonic()-start,history=history)


def check_offset_gradients(model, initial_offset, seed=281080):
    rng=np.random.default_rng(seed)
    # A central finite difference across an L1 kink is not a derivative test.
    # Test a nearby interior fixture; the source itself is audited separately.
    p=np.asarray(initial_offset,dtype=np.float64).ravel().copy()
    p+=rng.normal(scale=1e-3,size=p.shape)
    r=model.evaluate(p)
    direction=rng.normal(size=p.shape)
    direction/=np.linalg.norm(direction)
    safe=float(np.min(np.abs(p+model.sumw)/(np.abs(direction)+1e-30)))
    h=min(1e-5,.1*safe)
    plus,minus=model.evaluate(p+h*direction,False),model.evaluate(p-h*direction,False)
    checks={}
    for key,gradient in [('Lambda','gradient_lambda'),('E_gap','gradient_gap'),('error_1_sq','gradient_error1_sq')]:
        finite=(plus[key]-minus[key])/(2*h)
        analytic=float(np.dot(r[gradient],direction))
        checks[key]=dict(analytic=analytic,finite=finite,absolute_defect=abs(finite-analytic),step=h,fixture='Nearby interior point avoiding L1 kinks')
        assert abs(finite-analytic)<2e-5,(key,checks[key])
    return checks


def test_proximal_subproblem():
    class Fixture:
        sumw=np.zeros(8)
        l1w=np.array([1.,2.,.5,.7,1.3,.4,.8,1.1])
    model=Fixture()
    h=np.array([0.,.2,-.3,0.,.5,-.4,.1,0.])
    result=dict(gradient_lambda_smooth=np.array([1.,-.7,.5,.3,-.2,.9,-.1,.6]),
                gradient_gap=np.array([1.,2.,-1.,.3,.7,.2,-.8,1.3]),
                gradient_error1_sq=np.array([.5,-.7,.9,.3,-.5,1.2,.4,-.2]))
    rows=[];step=.1
    for active in [False,True]:
        x=proximal_tangent_step(model,h,result,step,active)
        assert x is not None
        c=np.stack([result['gradient_gap']]+([result['gradient_error1_sq']] if active else []))
        defect=float(np.max(np.abs(c @ (x-h))))
        z=h-step*result['gradient_lambda_smooth'];threshold=step*.5*model.l1w
        loss=lambda a:.5*float(np.sum((a-z)**2))+float(threshold @ np.abs(a))
        assert defect<1e-8 and loss(x)<=loss(h)+1e-12
        rows.append(dict(active_constraints=len(c),affine_defect=defect,subproblem_loss_decrease=loss(h)-loss(x)))
    return rows


def run_schedule(weights, h2_i, model_factory, pair_descent, best_pair,
                 cycles, pair_seconds, offset_seconds, target_gap, callback=None, seed=282800):
    """Alternate exact augmented gauges and one-body offset refinement.

    Exact gauge rotations preserve the current represented model. The offset
    stage then spends one-body residual slack while preserving the two-body
    model and the original energy gap. This is block coordinate optimization,
    not a gauge invariance or global-optimality claim for the whole procedure.
    """
    ww,hh=weights.copy(),h2_i.copy();history=[];start=time.monotonic()
    for cycle in range(cycles):
        before=model_factory(ww).evaluate(hh-ww.sum(axis=1),False)
        a=np.concatenate([ww,hh[:,None,:]],axis=1)
        rotated=pair_descent(a,pair_seconds,best_pair,seed=seed+cycle)
        ww,hh=rotated[:,:-1],rotated[:,-1]
        model=model_factory(ww);p=hh-ww.sum(axis=1)
        pair=model.evaluate(p,False)
        assert abs(pair['error_1']-before['error_1'])<1e-8
        assert abs(pair['E_gap']-target_gap)<1e-8
        assert pair['Lambda']<=before['Lambda']+1e-9
        offset,report=optimize_prox_offsets(model,p,seconds=offset_seconds,target_gap=target_gap)
        hh=offset+ww.sum(axis=1)
        final=model.evaluate(offset,False)
        assert final['Lambda']<=pair['Lambda']+1e-9
        assert final['error_1']<.02 and abs(final['E_gap']-target_gap)<1e-8
        row=dict(cycle=cycle+1,seconds=time.monotonic()-start,pair_lambda=pair['lambda_eff'],
                 offset_lambda=final['lambda_eff'],pair_cap_seconds=pair_seconds,
                 offset_cap_seconds=offset_seconds,offset_actual_seconds=report['seconds'],
                 offset_accepted=report['accepted'],offset_trials=report['trials'],
                 error_1=final['error_1'],E_gap=final['E_gap'],offset_history=report['history'])
        history.append(row)
        if callback:callback(row,ww,hh)
    return ww,hh,dict(cycles=cycles,pair_total_cap=cycles*pair_seconds,
                      offset_total_cap=cycles*offset_seconds,seconds=time.monotonic()-start,history=history)


In [ ]:
class RankAmplitudeModel:
    """Fixed one-body target; exact quadratic two-body-error model.

    Positive per-rank scales are standard reparametrizations, not exact
    Hamiltonian gauges. The residual Gram and physical error2 really change.
    """
    def __init__(self,w,h,x,metrics,ref,shift,official_maths):
        self.w=np.array(w);self.h=np.array(h);self.p=h-w.sum(axis=1)
        self.sumw=w.sum(axis=1);self.l1=np.abs(w).sum(axis=1)
        self.p2=np.sum(self.p*self.p,axis=1);self.base=metrics
        self.lam1=metrics['Lambda']-.25*np.sum((self.l1+abs(h))**2)
        n=x.shape[-1];delta=official_maths.reconstruct_h2(x)-ref['tei'];diag=np.arange(n)
        delta[:,:,diag,diag]-=.5*shift[:,:,None];delta[diag,diag,:,:]-=.5*shift[None,:,:]
        self.e20=float(np.sum(delta*delta));flat=x.reshape(8,60,n*n);D=delta.reshape(n*n,n*n)
        self.d=np.array([np.sum((flat[r]@D)*flat[r]) for r in range(8)])
        self.A=np.array([[np.sum((flat[i]@flat[j].T)**2) for j in range(8)] for i in range(8)])
        assert abs(np.sqrt(self.e20)-metrics['error_2'])<1e-10
    def evaluate(self,z):
        t=np.exp(z);u=t*t;delta=u-1;h=self.p/t[:,None]+t[:,None]*self.sumw
        v=t[:,None]*self.l1+abs(h);lam=self.lam1+.25*np.sum(v*v)
        gap=self.base['E_gap']+.5*np.sum(self.p2*(1/u-1));eff=np.sqrt(gap*(2*lam-gap))
        e2sq=self.e20+2*delta@self.d+delta@self.A@delta
        sign=np.where(abs(h)>1e-12,np.sign(h),0)
        gl=.5*np.sum(v*(t[:,None]*self.l1+sign*(-self.p/t[:,None]+t[:,None]*self.sumw)),axis=1)
        gg=-self.p2/u
        return dict(lambda_eff=float(eff),Lambda=float(lam),E_gap=float(gap),error_2=float(np.sqrt(max(e2sq,0))),error_2_sq=float(e2sq),
            gradient_eff=(gap*gl+(lam-gap)*gg)/eff,gradient_gap=gg,gradient_error2_sq=4*u*(self.d+self.A@delta))

def rank_epigraph_step(amp,u,r,radius,error2_cap=.0199995,gap_floor=3.9148981):
    """LP includes all480 absolute values, with a bounded8-variable move."""
    t=np.sqrt(u);h=amp.p/t[:,None]+t[:,None]*amp.sumw;v=t[:,None]*amp.l1+abs(h)
    J=(-amp.p/u[:,None]**1.5+amp.sumw/t[:,None])*.5;gg=-.5*amp.p2/u**2
    gs=(r['E_gap']*.25*np.sum(v*amp.l1/t[:,None],axis=1)+(r['Lambda']-r['E_gap'])*gg)/r['lambda_eff']
    weights=(r['E_gap']/r['lambda_eff']*.5*v).ravel();jac=np.zeros((480,8))
    jac[np.arange(480),np.repeat(np.arange(8),60)]=J.ravel()
    C=np.concatenate([gs,weights]);ineq=np.zeros((962,488));rhs=np.zeros(962)
    ineq[:480,:8]=jac;ineq[480:960,:8]=-jac;ineq[:480,8:]=-np.eye(480);ineq[480:960,8:]=-np.eye(480)
    rhs[:480]=-h.ravel();rhs[480:960]=h.ravel();ge=2*(amp.d+amp.A@(u-1))
    ineq[960,:8]=ge*1e6;rhs[960]=(error2_cap**2-r['error_2_sq'])*1e6
    ineq[961,:8]=-gg*1e6;rhs[961]=(r['E_gap']-gap_floor)*1e6
    bounds=[(max(-radius,np.exp(-.01)-u[i]),min(radius,np.exp(.01)-u[i])) for i in range(8)]+[(0,None)]*480
    # SciPy forwards this HiGHS option verbatim; both CPU workers are explicit.
    with warnings.catch_warnings(record=True):
        fit=linprog(C,A_ub=ineq,b_ub=rhs,bounds=bounds,method='highs-ds',options=dict(threads=2,dual_feasibility_tolerance=1e-9,primal_feasibility_tolerance=1e-9))
    return fit,float(weights@abs(h).ravel())

def refit_rank_amplitudes(amp,seconds=60,cpu_seconds=120,error2_cap=.0199995,gap_floor=3.9148981):
    """Optional reader search. The frozen verification run never calls this."""
    start=time.monotonic();cpu=time.process_time();u=np.ones(8);r=amp.evaluate(np.zeros(8));history=[];radius=1e-4
    for iteration in range(200):
        if time.monotonic()-start>seconds or time.process_time()-cpu>cpu_seconds:break
        fit,offset=rank_epigraph_step(amp,u,r,radius,error2_cap,gap_floor)
        if not fit.success or fit.fun-offset>=-1e-10:break
        step=fit.x[:8];delta=u-1;aa=float(step@amp.A@step);bb=float(2*step@(amp.d+amp.A@delta));cc=r['error_2_sq']-error2_cap**2
        alpha=1.
        if aa>0:alpha=min(1.,max(0.,.999*(-bb+np.sqrt(max(bb*bb-4*aa*cc,0)))/(2*aa)))
        accepted=False
        for backtrack in range(30):
            trial=u+alpha*step;m=amp.evaluate(.5*np.log(trial))
            if m['error_2']<=error2_cap and m['E_gap']>=gap_floor and m['lambda_eff']<r['lambda_eff']-1e-10:
                u=trial;r=m;accepted=True;break
            alpha*=.5
        history.append(dict(iteration=iteration+1,seconds=time.monotonic()-start,accepted=accepted,**{k:v for k,v in r.items() if not k.startswith('gradient')}))
        if not accepted:radius*=.25
        if radius<1e-10:break
    t=np.sqrt(u);return amp.w*t[:,None,None],amp.p/t[:,None]+t[:,None]*amp.sumw,history

def test_rank_model(amp,numerical):
    controls=[]
    for seed in [301001,301002]:
        z=np.random.default_rng(seed).normal(0,2e-5,8);t=np.exp(z);m=amp.evaluate(z)
        native=numerical(amp.w*t[:,None,None],amp.p/t[:,None]+t[:,None]*amp.sumw)
        parity={k:abs(m[k]-native[k]) for k in ['lambda_eff','Lambda','E_gap','error_2']};assert max(parity.values())<1e-8
        assert abs(native['error_1']-amp.base['error_1'])<1e-10
        direction=np.random.default_rng(seed+1).normal(size=8);direction/=np.linalg.norm(direction);eps=1e-7
        plus=amp.evaluate(z+eps*direction);minus=amp.evaluate(z-eps*direction);derivatives={}
        for key,grad in [('lambda_eff','gradient_eff'),('E_gap','gradient_gap'),('error_2_sq','gradient_error2_sq')]:
            finite=(plus[key]-minus[key])/(2*eps);analytic=float(m[grad]@direction);assert abs(finite-analytic)<2e-5
            derivatives[key]=dict(finite=finite,analytic=analytic,absolute_defect=abs(finite-analytic))
        controls.append(dict(seed=seed,official_parity=parity,derivatives=derivatives))
    with warnings.catch_warnings(record=True):
        f=linprog([0,1],A_ub=[[1,-1],[-1,-1]],b_ub=[2,-2],bounds=[(-1,1),(0,None)],method='highs-ds',options={'threads':2})
    assert f.success and np.max(abs(f.x-[1,1]))<1e-10
    return dict(two_full_precision_fixtures=controls,epigraph_fixture=dict(passed=True,solution=f.x.tolist()))

def smooth_rank_sqp_control(amp,grouped=False):
 CAP2=.0199995;FLOOR=3.9148981;base=amp.evaluate(np.zeros(8))
 class Budget(Exception):pass
 start=time.monotonic();cpu=time.process_time();best=np.zeros(8);bm=amp.evaluate(best);last=None;cache=None;history=[];evaluations=0
 def unpack(v):return np.repeat(v[0],8) if grouped else v
 def reduce(g):return np.array([g.sum()]) if grouped else g
 def ev(v):
  nonlocal best,bm,last,cache,evaluations
  if time.monotonic()-start>60 or time.process_time()-cpu>120:raise Budget()
  if last is None or not np.array_equal(v,last):
   last=v.copy();cache=amp.evaluate(unpack(v));evaluations+=1
   if cache['error_2']<=CAP2 and cache['E_gap']>=FLOOR and cache['lambda_eff']<bm['lambda_eff']-1e-10:best=unpack(v).copy();bm=cache
  return cache
 def cb(v):
  m=ev(v);history.append(dict(iteration=len(history)+1,seconds=time.monotonic()-start,feasible=m['error_2']<=CAP2 and m['E_gap']>=FLOOR,**{k:v for k,v in m.items() if not k.startswith('gradient')}))
 cons=[dict(type='ineq',fun=lambda v:1-ev(v)['error_2_sq']/CAP2**2,jac=lambda v:-reduce(ev(v)['gradient_error2_sq'])/CAP2**2),dict(type='ineq',fun=lambda v:ev(v)['E_gap']-FLOOR,jac=lambda v:reduce(ev(v)['gradient_gap']))]
 try:
  fit=minimize(lambda v:ev(v)['lambda_eff']-base['lambda_eff'],np.zeros(1 if grouped else 8),jac=lambda v:reduce(ev(v)['gradient_eff']),bounds=[(-.005,.005)]*(1 if grouped else 8),constraints=cons,method='SLSQP',callback=cb,options=dict(maxiter=1000,ftol=1e-14));ev(fit.x);status=str(fit.message)
 except Budget:status='budget reached'
 return best,bm,dict(status=status,history=history,evaluations=evaluations,wall_seconds=time.monotonic()-start,cpu_seconds=time.process_time()-cpu)


## 3. Frozen same-start experiment and negative results

Both rank methods had maximum 60 wall / 120 CPU seconds, two threads, the same scored source, bounds $|\log t|\le0.005$, and the same accuracy/gap caps. Smooth rank8 SQP stopped after 65 evaluations with “Positive directional derivative for linesearch”; none of its proposed improvements was strictly feasible, so the selected control is the original source. The LP epigraph method accepted 11 steps in about 0.0815 wall / CPU seconds and stopped when no strict feasible nonlinear gain remained. Actual time includes line-search rejection; this is not a convergence claim. Exact endpoints and measured histories are frozen below.

Earlier bounded screens are retained: approximate KKT reopening found no qualifying zero coefficient, metric-scaled SQP gave no gain, and eight-rank smooth SQP did not beat a single global scale. Their failure is not evidence of a global optimum. The LP hypothesis changes how L1 kinks are represented, rather than rerunning the same failed method. These are local optimization records, not native verifier receipts.


In [ ]:
NEGATIVE_SCREENS=json.loads('{"kkt_orthant":{"status":"NEGATIVE_NO_QUALIFYING_KKT_ORTHANT","diagnostic":{"multiplier_error":0.005407523825553104,"multiplier_gap":5.909136224910275,"active_residual_norm":0.08996741224871775,"active_count":388,"zero_count":92,"max_zero_violation":-0.007800380993748378,"selected":[],"multipliers_approximate":true,"zero_test":"abs(smooth Lagrangian derivative)>L1 threshold+1e-7"},"native_checks":false,"new_competition_score":null},"metric_sqp":{"status":"NEGATIVE_NO_GAIN_BEYOND_CONTROL","control":{"status":"Optimization terminated successfully","metrics":{"error_1":0.01999949999999775,"error_1_sq":0.0003999800002499101,"Lambda":300.63788313261256,"E_gap":3.9148981000118965,"lambda_eff":48.359145234396465},"wall_seconds":0.037678792024962604,"cpu_seconds":0.03757900000000003,"callback_count":1,"evaluations":12,"history":[{"iteration":1,"seconds":0.014972499979194254,"feasible":false,"error_1":0.1327437384079486,"error_1_sq":0.017620900086517885,"Lambda":300.6970347388261,"E_gap":4.048437070127989,"lambda_eff":49.176378495347414}],"variable_scales":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0],"constraint_scales":[1.0,1.0],"gain_vs_initial_ppm":0.0,"endpoint_sha256":"dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6"},"candidate":{"status":"Optimization terminated successfully","metrics":{"error_1":0.01999949999999775,"error_1_sq":0.0003999800002499101,"Lambda":300.63788313261256,"E_gap":3.9148981000118965,"lambda_eff":48.359145234396465},"wall_seconds":0.1107984579866752,"cpu_seconds":0.10802699999999998,"callback_count":3,"evaluations":34,"history":[{"iteration":1,"seconds":0.01747087499825284,"feasible":false,"error_1":0.11053388614910879,"error_1_sq":0.012217739987224144,"Lambda":300.67086717041616,"E_gap":4.00978831934583,"lambda_eff":48.940521655669656},{"iteration":2,"seconds":0.05462516698753461,"feasible":false,"error_1":0.7383662898560978,"error_1_sq":0.545184777995859,"Lambda":301.89702554598983,"E_gap":5.549918168294198,"lambda_eff":57.62122857409907},{"iteration":3,"seconds":0.08607999997911975,"feasible":false,"error_1":0.14444733197760112,"error_1_sq":0.020865031715447305,"Lambda":300.7386157388058,"E_gap":4.050057339926525,"lambda_eff":49.18957523386326}],"variable_scales":[1.0424508363852563,1.1787023842424786,1.0831950045155596,1.1014062429734948,1.0086884469888553,1.1259901170320108,0.7798596963640748,1.0421140630861594,1.108317612565213,1.1337874217137036,1.0161147920648521,0.9906406218658207,1.1213624402043854,0.9879140131672772,0.8899062456197341,1.064679775549829,1.1260533206769883,0.7354520629929812,0.9530205820042076,1.0663417041976113,1.1112522802438178,1.085717839225433,0.9132694041733631,1.0599346642283864,0.9979395835250896,1.0513431132809565,1.054622310765607,1.0280421517984346,1.176609074770529,1.0700747746167352,1.0738319337094087,1.1232403576302057,0.9696816779960146,0.9965755059109446,0.9450948327220531,1.0886504011318021,1.1390949208179753,1.1004975672165196,1.0855379105196972,0.9838612131899892,1.0127518155276267,1.058212437947535,1.0316851645440746,0.9357975934935874,0.9349851310719836,1.1215999869613509,1.104300917008292,1.0830634229217213,1.0684318092532643,1.0846494092871235,0.985685086685482,1.1314951996281815,1.1170444871355285,1.0490867580490668,1.0909078534819567,1.0727732265803531,1.1138557571477379,1.007298923481146,1.0166269256622433,0.9861914553123506,1.0935468529254768,0.9842444212220163,1.0303473345475769,1.012919727762469,0.7276014831007281,0.7378750205063233,0.9877280744011431,1.1189290399139133,0.7722007946098037,1.0009125259048481,0.9623611328407993,0.9308976464658625,0.9448764152842906,0.8966507794923657,1.081333366388639,1.0694578635330287,0.985908795531955,0.9008636055132044,1.151739283788778,0.9935660357416046,0.7485851991147426,0.8280861140278494,0.8888295576262598,0.8936754235933136,0.9779252498918294,1.093115583131554,0.9923548030539528,1.0627355998047274,1.0105369708949026,0.9452733094447301,1.095638582082971,1.0655313368785715,1.0119338953722499,1.0539259402429102,0.978681534157318,0.9151615524122616,1.1300088242804025,0.9827623377494856,1.007856161095271,0.9671673943599006,1.044136604025385,1.0698830286208774,1.089136272310573,1.0715534863799852,0.9677316656035093,1.082366034320137,0.9801021432230699,1.142807942273557,1.0615564529837629,0.8652673659086619,1.0241519556822671,1.0307553505989906,1.094645649854454,1.0750009239483747,0.9823051547846444,1.0483495363460626,1.1122312152332363,0.9561071465141325,1.1549403114339392,1.202068933414102,1.125122137957597,0.814301093080301,1.060218131728697,0.8687070511285723,0.994325623375239,1.115166254591197,1.0627248748790685,1.0147868636200004,1.0986653176675523,0.9180105701177537,1.1161304852000524,1.0760090101446154,0.974476916607463,1.105680079165942,1.193787828236329,0.99708648546128,1.1325807942774542,0.678464614731661,0.792231084792706,1.1457239922397962,0.813639508130136,1.1032471621473317,0.9123893285781259,0.9680463454800085,1.033512169841185,1.0437530788367748,0.8117703230139015,1.0639636940028356,1.0637268112405918,1.13308772086946,0.8818665658118143,1.0228851444980795,1.0063546631543696,1.015161338187813,1.0271376393342972,0.9868959307226116,1.0598087201334896,1.0318845911431416,0.9505505413040731,0.9896016994562145,0.9488473736327956,1.132881102495635,0.9879564859482015,0.9735980262077275,1.0096285418409114,1.1763349358202995,1.0041954757826301,1.0717896099794622,0.9145740259598557,0.8655345395781213,1.0387965846767355,1.0546939543421827,1.0485937505662095,1.0271978787233496,0.9344410296054427,1.0978697011426148,0.9957708834191036,0.9262346173456479,0.8874669486907496,0.9381871497489912,1.0908689091213564,0.9556502396787453,0.9837840080470744,0.9992779809362824,0.7938277231813976,1.1025094175488126,1.1379004217085813,0.818959978793361,1.1080793321370275,0.9722277117920877,1.1022517005593326,1.0651469385879369,1.1851440601883039,0.9084605037084109,0.8094568469019723,0.8048335704339078,0.8000271688063152,1.0623504363679503,1.1201993302247444,1.0433104984536787,1.1359629490236096,1.1694043071979903,0.9672484708036048,1.1010104701836085,1.2095329308845177,1.1428353280540928,0.9710714942334716,1.1593055845696303,1.0745414359105379,1.142764019726454,1.1042039929732876,0.8327807834551484,0.7214039636211362,0.9244662326241752,1.0111642518768955,1.017130446211581,0.7702305513374618,1.0656615731437087,0.8941771044869451,0.7289100626619968,1.1115203463983554,0.8529334555389966,1.00374479368141,1.0022332377801129,0.7465532215237803,0.7966996818077537,0.8338613123923063,0.8668076361576146,0.7055366792480583,0.8970523675860309,0.9772809764995953,1.0778682147668714,1.029859498565629,0.8576057966404338,0.9790330816088579,0.9822871365688477,1.0550251419412349,1.0568066413047128,0.895430208467372,1.096970969615972,1.1676980078080328,1.026307454843725,0.8682487178440064,1.034082584041832,0.9900242270312215,1.0352605732424598,1.0596248581887282,1.0014473856823751,1.175643283220966,1.0549371856195677,1.026787644366004,1.016913749976468,0.9577377223830879,0.7953763670610392,1.136837262845495,1.1031359003384413,1.0412127349696099,1.0984097309643177,1.0722467693260396,1.1164158082910323,1.043325707024791,0.9448870559291074,1.0335595699556406,1.0806395882662243,0.8550479015426152,1.1018463268569025,1.0208683715373663,1.070480371403923,1.0510718093391016,1.0586566068246506,1.0579496683994445,0.8707759673998823,0.9473320021011867,0.9495538783444052,1.0916248317192245,0.8657459798968486,1.183937000570694,1.0972506293703552,0.669808356779074,1.0233340354209997,1.0945396831183598,0.7530730661784569,1.043914944023446,1.067573112089877,0.981854915019569,0.9823219718085334,0.8572846399128916,0.9043453244027501,1.0271227291187526,1.1465901753734034,1.2010756086418026,0.9098051235501591,1.0139424481143928,0.8901638157374238,1.0131153204460908,1.0258605461259585,1.1157105011148443,1.09741610839733,1.0783720020357794,1.0563959204853575,1.049185660042564,0.9442470684712165,0.9852995462460674,1.0585693443678779,1.0274374665505783,0.9497637334196093,0.7302499232363548,0.978412882663245,1.1639036264786742,0.9893036860677052,1.064913043591264,0.9645227107495582,1.0168309145481411,1.030627171417725,1.0131581408418162,0.9586937923699259,0.8439245233831566,0.9861865465466254,1.1283675371487991,1.0744252730365984,1.051981767787122,0.8633037582355496,0.9977448318913001,0.8457426465477926,0.849748276544058,1.1040629751943014,1.0423384328708978,1.2059492142837314,1.0292831517236514,0.9816061698396219,1.093864540330229,0.9736991537437597,0.6342026114329191,1.1249474592417572,0.9552754071896732,1.032635699182725,0.8639987875572664,1.1543032984026058,0.9606877542148206,1.0132410805274281,1.0694047881685105,0.8239286293434426,1.0001455337937746,1.0430690700177978,1.126733506284671,0.9998962179549115,1.0806981639770907,0.925494270067124,0.7525177856285238,0.7758206363292639,0.805951053266448,1.021714494917346,1.0375977510403955,0.9841783530459144,0.7706975806162774,0.7795099652494197,1.152089347519204,1.034115133572714,1.0557831368762483,0.9759061705827138,1.0224618758282582,1.0065550868257849,1.0363172413027817,1.008085629341356,1.0199055165716073,1.112812226285959,1.1125228990968707,0.8541845797248196,1.1339396935512278,1.0021114505692161,1.0980076635806546,1.0822819089597289,0.9981300011801028,1.0250617611174657,1.0126410931386298,1.1253480746500193,0.9960860259890985,0.9587182521719609,0.9701354595488458,1.047541812195075,0.9986606160739507,1.0479716284093,0.754829052648909,1.1256040769614135,1.012376017191198,1.012701837566165,1.0138610106952008,1.0891617582798216],"constraint_scales":[0.026196972998559602,0.7871126573812794],"gain_vs_initial_ppm":0.0,"endpoint_sha256":"dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6"},"gain_beyond_control_ppm":0.0,"native_checks":false,"new_competition_score":null},"rank_amplitude":{"status":"NEGATIVE_NO_GAIN_BEYOND_CONTROL","control":{"status":"Optimization terminated successfully","metrics":{"lambda_eff":48.359145234396465,"Lambda":300.63788313261256,"E_gap":3.9148981000118965,"error_2":0.01999939507696627,"error_2_sq":0.0003999758034445827},"log_scales":[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],"wall_seconds":0.004120540979783982,"cpu_seconds":0.0013139999999998153,"callback_count":1,"evaluations":1,"history":[{"iteration":1,"seconds":0.003128332959022373,"feasible":true,"lambda_eff":48.359145234396465,"Lambda":300.63788313261256,"E_gap":3.9148981000118965,"error_2":0.01999939507696627,"error_2_sq":0.0003999758034445827}],"gain_vs_initial_ppm":0.0,"factor_sha256":"dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6"},"candidate":{"status":"Positive directional derivative for linesearch","metrics":{"lambda_eff":48.359145234396465,"Lambda":300.63788313261256,"E_gap":3.9148981000118965,"error_2":0.01999939507696627,"error_2_sq":0.0003999758034445827},"log_scales":[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],"wall_seconds":0.0028547500260174274,"cpu_seconds":0.00263100000000005,"callback_count":8,"evaluations":65,"history":[{"iteration":1,"seconds":0.0002590420190244913,"feasible":false,"lambda_eff":48.483697233817225,"Lambda":300.6486908622351,"E_gap":3.9350807603131632,"error_2":0.08569950692975914,"error_2_sq":0.007344405488003835},{"iteration":2,"seconds":0.0005936669767834246,"feasible":false,"lambda_eff":48.46292700656215,"Lambda":300.66080658124866,"E_gap":3.931527068687502,"error_2":0.08147473565543142,"error_2_sq":0.006638132550122429},{"iteration":3,"seconds":0.0008472499903291464,"feasible":false,"lambda_eff":48.468341917648075,"Lambda":300.6658694790015,"E_gap":3.932344407569903,"error_2":0.07703996572460449,"error_2_sq":0.005935156318848234},{"iteration":4,"seconds":0.0011188340140506625,"feasible":false,"lambda_eff":48.4528788316655,"Lambda":300.6528662430684,"E_gap":3.9299913023312816,"error_2":0.07332251243864775,"error_2_sq":0.005376190830315653},{"iteration":5,"seconds":0.0013973339810036123,"feasible":false,"lambda_eff":48.417560607330316,"Lambda":300.64645078267773,"E_gap":3.9243110744035876,"error_2":0.06069957510991339,"error_2_sq":0.003684438418524017},{"iteration":6,"seconds":0.00162583397468552,"feasible":false,"lambda_eff":48.49578245261161,"Lambda":300.6665103281714,"E_gap":3.936819334450032,"error_2":0.08320765811950168,"error_2_sq":0.006923514369731872},{"iteration":7,"seconds":0.001866958977188915,"feasible":false,"lambda_eff":48.47223876100562,"Lambda":300.6606235228497,"E_gap":3.9330504753743387,"error_2":0.08168130369385064,"error_2_sq":0.006671835373127059},{"iteration":8,"seconds":0.0020539999823085964,"feasible":false,"lambda_eff":48.359145157722224,"Lambda":300.6378830296423,"E_gap":3.9148980888742946,"error_2":0.019999395145673327,"error_2_sq":0.00039997580619278186}],"gain_vs_initial_ppm":0.0,"factor_sha256":"dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6"},"gain_beyond_control_ppm":0.0,"native_checks":false,"new_competition_score":null}}')
FROZEN_MANIFEST=json.loads('{"source_version":363660552,"source_factor_sha256":"dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6","payload_sha256":"6795bc0ca52c09b49e22293894794333d24ebbcb10cde62c2b85a6c056a9fbc4","arrays":{"rank_sqp_control":{"W_sha256":"770e778d82ff72097882fcb790946443145b1db594115bfe73cf5bcff18bf8b5","H2_I_sha256":"a60951dd57d4ae756fe7309a1ece640fba67897c4547425b9b0179dbeb65fe38","factor_sha256":"dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6"},"epigraph_rank":{"W_sha256":"041b7096cffd08ee9ef28a5fce3d45e6c0bb14b4f288548ec16352b8b98212c3","H2_I_sha256":"66633a788f3321ba3f81502f9e62ea4083d578843efa4c8acb4e7e6a04ab1d39","factor_sha256":"a75124d9c4bdede37fdd2d65c5ead872badc514945bb081de9d439b1cb091158"}},"historical":{"venue":"macOS CPU, two threads, 30 October 2026","maximum_wall_seconds_per_arm":60,"maximum_cpu_seconds_per_arm":120,"arms":{"rank_sqp_control":{"status":"Positive directional derivative for linesearch","metrics":{"lambda_eff":48.359145234396465,"Lambda":300.63788313261256,"E_gap":3.9148981000118965,"error_2":0.01999939507696627,"error_2_sq":0.0003999758034445827},"log_scales":[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],"wall_seconds":0.0028547500260174274,"cpu_seconds":0.00263100000000005,"callback_count":8,"evaluations":65,"history":[{"iteration":1,"seconds":0.0002590420190244913,"feasible":false,"lambda_eff":48.483697233817225,"Lambda":300.6486908622351,"E_gap":3.9350807603131632,"error_2":0.08569950692975914,"error_2_sq":0.007344405488003835},{"iteration":2,"seconds":0.0005936669767834246,"feasible":false,"lambda_eff":48.46292700656215,"Lambda":300.66080658124866,"E_gap":3.931527068687502,"error_2":0.08147473565543142,"error_2_sq":0.006638132550122429},{"iteration":3,"seconds":0.0008472499903291464,"feasible":false,"lambda_eff":48.468341917648075,"Lambda":300.6658694790015,"E_gap":3.932344407569903,"error_2":0.07703996572460449,"error_2_sq":0.005935156318848234},{"iteration":4,"seconds":0.0011188340140506625,"feasible":false,"lambda_eff":48.4528788316655,"Lambda":300.6528662430684,"E_gap":3.9299913023312816,"error_2":0.07332251243864775,"error_2_sq":0.005376190830315653},{"iteration":5,"seconds":0.0013973339810036123,"feasible":false,"lambda_eff":48.417560607330316,"Lambda":300.64645078267773,"E_gap":3.9243110744035876,"error_2":0.06069957510991339,"error_2_sq":0.003684438418524017},{"iteration":6,"seconds":0.00162583397468552,"feasible":false,"lambda_eff":48.49578245261161,"Lambda":300.6665103281714,"E_gap":3.936819334450032,"error_2":0.08320765811950168,"error_2_sq":0.006923514369731872},{"iteration":7,"seconds":0.001866958977188915,"feasible":false,"lambda_eff":48.47223876100562,"Lambda":300.6606235228497,"E_gap":3.9330504753743387,"error_2":0.08168130369385064,"error_2_sq":0.006671835373127059},{"iteration":8,"seconds":0.0020539999823085964,"feasible":false,"lambda_eff":48.359145157722224,"Lambda":300.6378830296423,"E_gap":3.9148980888742946,"error_2":0.019999395145673327,"error_2_sq":0.00039997580619278186}],"gain_vs_initial_ppm":0.0,"factor_sha256":"dacfed3f0fe6d90d7f13813a07300e6a43bebe94c3f6a4b5a34c2005e1c314c6"},"epigraph_rank":{"status":"PROMISING_EPIGRAPH_AMPLITUDE_GAIN","termination":"No strict feasible nonlinear gain","metrics":{"lambda_eff":48.359143600577994,"Lambda":300.6378611691011,"E_gap":3.9148981235181517,"error_2":0.0199995,"error_2_sq":0.00039998000025000003},"gain_vs_initial_ppm":0.03378509816354125,"gain_beyond_control_ppm":0.03378509816354125,"rank_scales":[1.000000000000396,0.9999962184518255,0.999995332909997,1.0000037817301808,0.9999962180409346,1.0000042072038702,1.0000019870532302,0.9999964166612779],"accepted_steps":11,"history":[{"iteration":1,"predicted_descent":-2.0271847626318618e-05,"accepted":true,"alpha":0.07689822982468646,"radius":0.0001,"seconds":0.012766124971676618,"lambda_eff":48.3591438244805,"Lambda":300.6378638881727,"E_gap":3.914898124134366,"error_2":0.019999499792929313,"error_2_sq":0.0003999799919673796},{"iteration":2,"predicted_descent":-1.326221551778417e-05,"accepted":true,"alpha":0.015712955635648763,"radius":0.0001,"seconds":0.01602608396206051,"lambda_eff":48.35914361291712,"Lambda":300.6378612896377,"E_gap":3.914898123938817,"error_2":0.019999499999589273,"error_2_sq":0.0003999800002335714},{"iteration":3,"predicted_descent":-1.3148304986998482e-05,"accepted":true,"alpha":0.0006853453429289967,"radius":0.0001,"seconds":0.01915191696025431,"lambda_eff":48.359143603682455,"Lambda":300.6378611767828,"E_gap":3.9148981239227587,"error_2":0.01999949999999918,"error_2_sq":0.0003999800002499673},{"iteration":4,"predicted_descent":-1.3143020824557539e-05,"accepted":true,"alpha":1.5268074795786633e-05,"radius":0.0001,"seconds":0.022190416988451034,"lambda_eff":48.3591436034764,"Lambda":300.6378611742692,"E_gap":3.9148981239223404,"error_2":0.019999499999999424,"error_2_sq":0.00039998000024997694},{"iteration":5,"predicted_descent":-1.3142902414386981e-05,"accepted":true,"alpha":2.56352097167268e-05,"radius":0.0001,"seconds":0.02515383396530524,"lambda_eff":48.359143603131336,"Lambda":300.6378611700487,"E_gap":3.9148981239217875,"error_2":0.01999949999999995,"error_2_sq":0.000399980000249998},{"iteration":6,"predicted_descent":-1.3142704498037006e-05,"accepted":false,"alpha":6.948223266057271e-15,"radius":0.0001,"seconds":0.029033791972324252,"lambda_eff":48.359143603131336,"Lambda":300.6378611700487,"E_gap":3.9148981239217875,"error_2":0.01999949999999995,"error_2_sq":0.000399980000249998},{"iteration":7,"predicted_descent":-3.532875725298368e-06,"accepted":false,"alpha":1.9478629988501355e-14,"radius":2.5e-05,"seconds":0.032854333985596895,"lambda_eff":48.359143603131336,"Lambda":300.6378611700487,"E_gap":3.9148981239217875,"error_2":0.01999949999999995,"error_2_sq":0.000399980000249998},{"iteration":8,"predicted_descent":-1.0700113275419199e-06,"accepted":false,"alpha":3.1344688780599023e-14,"radius":6.25e-06,"seconds":0.03623095899820328,"lambda_eff":48.359143603131336,"Lambda":300.6378611700487,"E_gap":3.9148981239217875,"error_2":0.01999949999999995,"error_2_sq":0.000399980000249998},{"iteration":9,"predicted_descent":-3.8029094895364324e-07,"accepted":false,"alpha":1.1387608193218553e-13,"radius":1.5625e-06,"seconds":0.03971549996640533,"lambda_eff":48.359143603131336,"Lambda":300.6378611700487,"E_gap":3.9148981239217875,"error_2":0.01999949999999995,"error_2_sq":0.000399980000249998},{"iteration":10,"predicted_descent":-2.058412942318455e-07,"accepted":false,"alpha":4.55187967181196e-13,"radius":3.90625e-07,"seconds":0.04312983399722725,"lambda_eff":48.359143603131336,"Lambda":300.6378611700487,"E_gap":3.9148981239217875,"error_2":0.01999949999999995,"error_2_sq":0.000399980000249998},{"iteration":11,"predicted_descent":-1.6223093624034846e-07,"accepted":false,"alpha":1.8155560693900237e-12,"radius":9.765625e-08,"seconds":0.04661266697803512,"lambda_eff":48.359143603131336,"Lambda":300.6378611700487,"E_gap":3.9148981239217875,"error_2":0.01999949999999995,"error_2_sq":0.000399980000249998},{"iteration":12,"predicted_descent":-1.5131347197439027e-07,"accepted":true,"alpha":0.0039462382407432,"radius":2.44140625e-08,"seconds":0.049458791967481375,"lambda_eff":48.359143602534,"Lambda":300.6378611698283,"E_gap":3.914898123827343,"error_2":0.019999499999999986,"error_2_sq":0.0003999800002499994},{"iteration":13,"predicted_descent":-1.5073046455427175e-07,"accepted":true,"alpha":0.0021850692366464147,"radius":2.44140625e-08,"seconds":0.0523026249720715,"lambda_eff":48.35914360220466,"Lambda":300.6378611697063,"E_gap":3.914898123775278,"error_2":0.019999499999999996,"error_2_sq":0.00039998000024999987},{"iteration":14,"predicted_descent":-1.5040906831131906e-07,"accepted":true,"alpha":0.0018656984042609188,"radius":2.44140625e-08,"seconds":0.05538216698914766,"lambda_eff":48.35914360192428,"Lambda":300.6378611696023,"E_gap":3.914898123730955,"error_2":0.019999499999999972,"error_2_sq":0.00039998000024999894},{"iteration":15,"predicted_descent":-1.5013548271269883e-07,"accepted":true,"alpha":0.0028695403364652175,"radius":2.44140625e-08,"seconds":0.05832749995170161,"lambda_eff":48.35914360149343,"Lambda":300.63786116944226,"E_gap":3.9148981236628466,"error_2":0.01999949999999999,"error_2_sq":0.0003999800002499996},{"iteration":16,"predicted_descent":-1.4971505457594958e-07,"accepted":true,"alpha":0.0034872420925532036,"radius":2.44140625e-08,"seconds":0.061465041944757104,"lambda_eff":48.35914360097146,"Lambda":300.63786116924797,"E_gap":3.9148981235803406,"error_2":0.019999499999999993,"error_2_sq":0.00039998000024999976},{"iteration":17,"predicted_descent":-1.492057535301683e-07,"accepted":true,"alpha":0.0026370546622413093,"radius":2.44140625e-08,"seconds":0.06446041696472093,"lambda_eff":48.359143600577994,"Lambda":300.6378611691011,"E_gap":3.9148981235181517,"error_2":0.0199995,"error_2_sq":0.00039998000025000003},{"iteration":18,"predicted_descent":-1.488218828171739e-07,"accepted":false,"alpha":8.124313257030739e-16,"radius":2.44140625e-08,"seconds":0.06779308395925909,"lambda_eff":48.359143600577994,"Lambda":300.6378611691011,"E_gap":3.9148981235181517,"error_2":0.0199995,"error_2_sq":0.00039998000025000003},{"iteration":19,"predicted_descent":-1.4609000587029186e-07,"accepted":false,"alpha":1.453284372284975e-14,"radius":6.103515625e-09,"seconds":0.07114524999633431,"lambda_eff":48.359143600577994,"Lambda":300.6378611691011,"E_gap":3.9148981235181517,"error_2":0.0199995,"error_2_sq":0.00039998000025000003},{"iteration":20,"predicted_descent":-1.4540634296622557e-07,"accepted":false,"alpha":2.2945305537876823e-13,"radius":1.52587890625e-09,"seconds":0.07455091696465388,"lambda_eff":48.359143600577994,"Lambda":300.6378611691011,"E_gap":3.9148981235181517,"error_2":0.0199995,"error_2_sq":0.00039998000025000003},{"iteration":21,"predicted_descent":-8.69261818081668e-08,"accepted":false,"alpha":2.0698559359425618e-12,"radius":3.814697265625e-10,"seconds":0.07798908394761384,"lambda_eff":48.359143600577994,"Lambda":300.6378611691011,"E_gap":3.9148981235181517,"error_2":0.0199995,"error_2_sq":0.00039998000025000003}],"wall_seconds":0.08146866696188226,"cpu_seconds":0.08146900000000001,"solver_warnings":["Unrecognized options detected: {\'threads\': 2}. These will be passed to HiGHS verbatim."],"factor_sha256":"a75124d9c4bdede37fdd2d65c5ead872badc514945bb081de9d439b1cb091158","native_checks":false,"new_competition_score":null}}}}')
FROZEN_BYTES=base64.b85decode('P)h>@EdT%j2mk;8Apk2H-t_<f|NsC0|Nk2R6aaEzZfjq2ad2N_Z*FvQZ){(4V_|G%b1rUhc>w?r0KfnM00000080P>000000GsR=>KhQr$WX>mt)7xvoLr=CrJ!z;W}&X5pq`drR8o|f7oT60k_r-cOUx-w1&SAEBo?Fs`5G2FnmP)#3KRe?1~B+wPc97rP)h>@EdT%j2mk;8AplRcCei=@|NsC0|Nj{P6aaEzZfjq2ad2N_Z*FvQZ){&kGG9q9Zg6=401yCx4*&oF0002s4FCWD0001;7x_O_YkxvFDO$Kx=uYukN)aW+>7iVSEVqPYO_sW3t=v*gv<Tf>p~Y<>ViJk6Hpe3dS;81I7&B%k47o@{)H`$LzW=~`etAC6_xpXm`*zZtk2tv=U4%P}yP$s}C?+ske>Yy=D%e8b2(KR;5gi*HaM~{-`b3ZvZWj<16C{Dhga$+gN$v&~MtE}*LnHh}{QvtYRhQ5&g<j)8qQi6J`0X^{s-IASOQOSsE!~{V%^Zl6d~328NW7;12LFyR@F{QF@tAQpbk8UgDRhit__w}v^t(sV%g(%UXufyK(xa~wNZ~kpJM_97MLD=c2<<ntR(`(U2t|}evtuPCz?bpzMYA;*^e*Rbt=GlmSXWPx5iK9Z@Hihj#@E&04TAB(5lQwZ7i<;Hh9T2_TDf4Z7mz&`78&L@gGuLqUayOO3)<=|TG<9%={v%LYa23fEAQ~Jp$Ui^G`wSL)dSkBUfs*T$OhuS-+%Zg>>~`tb=~^${2k<t-?|p^nUBfCA817ec9F6|HZ*}5`9HM++*x2`yMOO$-%nD!Nx6itp2Hu$;l}+?>xuTLIZXfyH!15})*6D=&sTfcJ$?c@|0+4p*Q%4*zs?x>j-!7~ccBAs$|GjLAH5*qvW{6NVFb<xk?56_ZW*1mBEs{wy2RMd0G?feJq6iA7+mRWAG+o^67tbY76%-TVbzv$Dr|JFA8`KO!4!A&0=qk+%<UE<@a=zq=9eQpnAM)UUR$9I5(ZcL6qVD##K<{?Kp_k4-d6^doOnQ>4sLYE4g$L$>D`Y4xR{(r{8m)zx4eWFG#MpQ`H%-Kst+nCgtkGi&0hkaJNCgR*I(Mq^SRJjy~44`saN{0xBo&1nsB6KQ)&PYh~E8D=99<-aRD3h3Rd+?^d;~NDk`LWad|^`kt`Am7Oh}{_&L4HmWO&_Rzq-FcEwYeJ7u@<yAKPquX0~85=K#Tj~b=GvlOk3L#RX!C-L6o8!BX+Py1T<Pzbe(ES835QlS8ps2Z#H!}?aML)$m<Fgs#RZ=;o6JXTiz5Zrn3d&M!L4DSESPK3FuSwp*b5Gd|A73|OJlfq3dE7l_<(R;FL2o35UOfuVx9|HsVmU1U11>kf~x<}J{3OK#_Gj(K!57ZjH!Zzl};-av=6xCh7X}wDi3n-zAdPpr7*7)98U(X%G>`-l~L#(`)ZGYs>l*O&M6AjZ({8f4H0IccB^fovr^E(cgWhFKoB;{tap~<>CnHe-XBwx?JI$}%)Oul9!$ra0wTBU+_Xr}D}k4sECuy6lRU*wC`2jS?}4+v?zP|2YTyK^#EbxG@`)=}Js&f!}U>?(RNI`*&qQaIsjZZi`6w4S*41Qq15@-|XAJm}!V&i?vJX4n1fPHDV2_b831U9LF5n$!d0FX=1a-O2?DO;@Av5k0^;+jd)pmJn8+xY_!4dR+RB@cLXOCO5)f7VlH;H?XtRtWA1mxt@BDn!fn-r$wdA?p_g!=JCz96u0+5&&_WB2fZ4=@|N{KQu8=q{5X%2Tq&t1`{gS3V}0;{ezM5Z^(b(~>L5!L(2uUE`6<E+bjk3#IIK^onG%tipcSf*$k;n;epx$OBd??J=)eS2T;}U^q>+W`JEiA}>DO_!6X6{cEsk6@jOmq-M?wY1-?X1e8N&FQT+G1y#^aP>=ey`eH2wY;FXl-e99W^8qnbGgd3WvGIy(7awRPH8^`s_PNF(G2RSSSfJAPLbjSbe-2hdZVvjJ`KWOcZa0LsyfZT6&7{*6Bl_oUl6>0^}ulV@r504YdYdd)1b7mM4ZUOqz5Ax$54V1ddZZn;lGKd7_6VI{9Y0|Vyar%o)N0ZB`4WtzY)sOWNKNvF~T##gA=iK^jjW3!jF0Sop5B`L52XlZv?*x-90ep?&)VgViadneKTGFTw@z}o7jwJacL(|KA{+zpL%d+sUs(g6OL4xNDVK<YQ#Osy6Zo?h$_|K-&v@Q>`upU#!_%cPGtkw(AWT~|nh(7dNQJfVdFCc89LpCUpSs(!6&nUc&7kX?@)Cp(P2`gsJZMKs=CWXSo~{;MSQ+=xvX5ggEYT?sZPP5|{)3Jx{5{>A?^N|GL<$8uzSq^1}V*S3>IYbjuLGz_Pw@E|oS-EUHp1E+r}74{{|>Y`nx1YN9HLQKfwKnrgTj++M??(zSSE!1HGE5Dr<_D!uYUc=_9<MAHYxWV~LT^$c_ys966?c~7^DuN>#{X#gnbtWK@HwBYIc5f6dFfe^y<~E{{$0(;AAG0But{<srN(18dagF^GvijL(Q!UkVYz%2ebsuIIZYv*#k#`m~c-PS(zR>>slGXvB=(?-wgd;;1Us^TVwk5F5=~^#%7F0I&sj3q`nl0T3xm?hGZi=9X?*s)VDJ#33d*NP>pzo=9Eg(579M=&w3DXOg?p#?&1JiUnzcqLojCH5K>iSBD&wR=c5mvKdfsS#+tP=%Z&8tvZ*3O303e0nTIW023s;omSQWV~Kn+(D^lE$3>a;%@@$PeD2vbeZf#UrY-y~8i0dg7zhsR-w=%~(QBqx4Q(z%^lXi<@f^^MdE)p5z9wu^?mQ(9vFScj;7l_U1mg?&#tnU%7s8yOI))XV9QR{S}JC@BNS&HpEhPZ3PD}#6G>zG>GL<mJJoj{m2ffjAj9omMarP>uz8>;U-^o3iA_=@bnNG=d6?X{#gfhS3I1GEGMO146N-1kDX}&SJF@zA98wgh!O>AD!<>|=)i}>1MUk){JGH9+J$)D<_jdu2#06)Vg1-OS-TILcWZv5BM&S-7;J9thLM~0%{V;hh9U=NL5J;tl+Hfay#*y!rQEgkqDb$x`O7L1w?(77TN$HLf30i=wMgll_I~$B4)hdnO;tD`nSU?N-}qUx6GU=SS^E1suzus3(23wz;Z?`NhCow=z(@Hp544efURU~%12PG=o^7H@DIG`oO*2YA_ecNml>ulUN}LxUR4DSuit!Z=K>wy~b4oFE;O;d0x#0{SGUoFlnHPpZB{4hTuXiIbcjj>G{T)JB-F~iqt)Hwu9kPniaa_>xWR;IFI5u#KoiPL^cZ38Jt2h83?Wi%9F@oixU0f|<R61&A@DCr>)~#|>{D%c}V}D%u@<XyW==|=Gu#XGWJ35qBJ>de$Oe@vW6u`&hx1Pi)b6~-!UwA}{Y|dqcyg-yncxh2CLLl~kA0+mX?2`-vqiY2=7s-F<(U91ihp21Kx!|_mpQ{EwGNDD`=O~e>0P4QoR(wp6hw({y{~Yx@9HX);i32>vPOndI5y0q8+x|A$!v&|e?;Sg8M1xn6Z267j{a|bC{^yK4l0CHM);mj+5%9oVEnWv<LZ+PIhLvY}u)JPBQibmR&HbCJkP5xY=?1}~ehj{tP><gJv%K1XFPSTE7BcX`gYYgR?26d~4or6|-ardrK__LW`rr0)p^;WOpL(_r=FS&zfFzI3q@kx?21)h{;Ytx<fCfH=dUoZ#8V6w^p7~l9gMcs<qGhF22jm2k=G#7yf!lnJ<J#pyXns*HF+ETSIICBC<IoXsHm!KRDT5DUC#rs)ci}^WSwf(N=R5fQmt!A~>e9if?t&2UWghr;(d6h~rjq?`k2dM{eI}e%SsABF9)_gdVP<@zA=o`+a=D~}2MZ?}6Q8H>;U1ThM&UZc(Asie05O{fLa9#((QboKT-<Z?Kw}?F!X+N}HI>wfv39}brTt*yQe4zBZ3Z~a==$VC<V*d7H!LF~)8g)`B>860;@)sF=z#zzn0JXcnD+v)<==%KO?0^D+s7NLcF5`&ck(1^G!}Df{ZNx^pMH50O?zp7GG+r0#JOZEn}#!Ca@6Nn88jNibH3XsuIq*d;EJ2a(|#~R<hD1<QL%bS9jQmvI*e2YyqH*==BkQheM!6oRg+sAWBis4;+Eio<R4*q9_xtjLg}IvvGSJ~Ags*i2I<2PtTmf?9U|!igwn{JXQ<T}-=e?<#DsR=H=Z~Q=r#6+hks!CR-30<iz-;oFa&W_m?~=6V!4YACN-}(KbaYXVP9u&PL>L#bSn=vR-;-jqzCK5Xh4u-CqFzz1+}J4Z^~6VfJxWzmf2TqSsi#ilkN)>S^1^tK2t)fT5KnnVA1S%ec(WmoO9F5HB9(`{8nxJ>o`WQkoOGTGrC`U`h@`C^v7S@STLY%!^I4CI}LD-c$VK2wt?;0Ir6hJlDSA|stX!s!_=tFsaErSz;!lfGI*2@uQe%#=<KDymWesdoSZ(;Nwnvb+c80ll3T-yH&o!6U@B%^XF(M|wdlEd9@sSZjcvJ10Nd-H;kr~Q@X7C;W}ExD&@Gbul68O!<kkue4pI7GX#8xT;eI;!G~=n4d4mG3{?sBaYUY5@n^9{P>u~}3zGhzytq1z6xW2ev+zw)w+KD5_DPaELe`aV$BzwHi%}ib=wg=!e?pL9qs`i>0hq$2oSA#@Yg!NTiY<?OR-;uOpB+21sdVg&fC_L}w?Us%86>Hn}_ZU9e<s^DwcK_nmWDdNobNL~l4#3KVO^FBVWb;eqWF|@|zBWV<QDCy4?hw~c_TQ8xx{mOUCK-x727o%()R#HWfpIa(>wo*&5B>{KO9KQg000080000X08|_-K>z>$|NsC0{}=!i0A+A#XL4b1XkT(+Zfjq2V_|G%b1rUhc>w?r0KfnM000000D1rb000000GsR=>KhQr$WX>mt)7xvoLr=CrJ!z;W}&X5pq`drR8o|f7oT60k_r-cOUx-w1&SAEBo?Fs`5G2FnmP)#3KRgY?`#Y}@WKA;&(nfG{=c_B%#mRB4M@LiQTxse6lZOm3>G)L8I#EZ<V!oHiZTId(Y2;EAaMXtO9KQg000080000X05hk>#sB~R|NsC0{}uog0A+A#XL4b1XkT(+ZfjpiGG9q9Zg6=401yCx4*&oF0001n4gdfE0001;R{1}a+aI4K$(Bl`NQy!f*>bx)N3vA9a?6%d$(ksaNkT%AETyqrlx$gxkRr=-WH;FcvlxS!F~%;+6>jPKJl+4``~32}KA-bh-skf<pZ7WO7S?}Qo?k=oBLpfryZAc!C>@no(s$KUQj=G5z3D^ralCQ)rjN4=t~Yh`^mW1XzV43RE|{#WrzWqfp`s>#SN{L~M0}!X!S3N~NRrM`57MTBm)y__LL?1ZuJ2=~s<I&g`%c9&4UnPBd#-fN0Lj}6<`>laVBytw(}G=7zt*d4X|*vsi(Yl-&O(2IufN;&mjPT)Kr9WCd%DG4OGlw$9=Y=K-_20|Nwe04(ozta|Ei$PkOP95+(QkDzw#Jv``l=dAtmhP??C$%_exKh!LK-(=M0kdei#dDO~BT<ui}L(3=lH)eT_<98+fsCzGSCQIWSDs`!uY~!TmzyP1+44TyxJ|7@LFo=_*f+_30pTx+p2{?l4$V`Pe$|*#QaKl-Qp+)lj!3Hr(y=$gh0T;-3tXH`HXf?P&p|wpdw5n?WGAdYQ5L()cfYK7Y0w=u6(Y<fT3U1Bkn(bg4W@nh{gDCp!iWBPDE2(_Vnaqon~Ovi1M%KV-YX-UjNogflcy|Kd5z@eTuc98=Kh4w;0fch}G=DSiLhWxCYLD(VL_wV9wK&52%^IrdB68QNcKrE1{TVgFwr2s<y;_E~v&POS_8IWHa7QWpc{J4~l)>rF!bPekS)Yc4FQ%9h(D_zo_8kaQ?|PXl`=P3j(w4g#y8dFt*zxS*J^seYEo2Ulz?rr$VnusjyGA9ePiSa7#*nwjX8YXVOHLN1gnyqKcq-VUEPv$<{N{g9L|UAT$If!;@5uD1Wd!2Aqd^I}|)+_+D&*^XS`E>fQS=spXSpJ{L9ZW_SwHuD7DZNTD6xt6U-Mx;fX8+aR8p#AsCbr;VvAYrFyQ)^{9WXO?@WjPFD_PT*2#iuCwR{YYVDGKCwJ+Ko)8NkyeNPTO0FN}G0w7xNA6rM<(JN!7c7fSlt>050ZfO3_W-A?S~VtClY70u64QvUq;s=P6H#Uw8MdOogS^$v&(L=GtmF6uD&fVZ#FHj&$p>2)8iYWpG#f1xPgMujKdTxU7T&j7MJ@yGoyJTPu~w2%Lh0?5l^MTBo7AS-U@x<NMH7ekX)rAV}1qNsvM9|SSQw#d0U4s0zy^_tI_!S>-Nt#m095fWO+M{QV`93|Gj_aSQ45;v`FGz2xGvW3BIyzUH>*~?5MlB4ZH$sC5?4{o_vO{Kx0KxxJ<jYeRpuz^Uo#LtJ+nZf4~sK%sG!!s5QSoP#o_FTf}!y~G0I0JR<%--Sq4wpsa)Pq700>QoS_v1wL%R7zP8x?d+ALL2Ws=)N+8_G77pmsslCC<*h;JaPd+F3Ri=F>B_?s<dnOUN@=vli>uSuL4ok!atEq8lzXbf9#~CO!8M2R!bY&NseE2Xg_Pel*!pD7&@QXJ%m*^YcF_=O2s3-*|E1&v@)lvN_ohjm04a+HUH`{<Gc*Rw3w>g!SI`mH57q8-tr(qVA_ZTXl5y!|%B^djjm6Kuxvv!~9$}IKAs~N?GL?WK6S7Txa?*{8@0__&a1M@jQT*_#B9JycoWZa@Sn7zXf_QnV(M74Z!l+44uzQMCY8Ovzi)lzd^~e!<mSmE_h#kat^xFjMe>{2eJIpQ!+KbVYpoPss_JDM5tHy#B7?t^naf>9j-&{ZnE~LMUP?rvsFr`TQIvh#4#)xi(i+qEc6DE9JiBPzQ%<MUXNdCr}80Vw}Ie7_Xtp=Gpn^CTOe=yxxA1n9_W6nt&mL}23eK*pekn=tgg=%4_D(s6`!Bg)(I4RzQxI5+31yni}jNqcv!wPtp>3;#P~?D*cB&iJ_s*WVjKbx-p~`W)Tu!*m#)ZCX&M0XcT=J#woyUEil&9B*dl1X$lEFldtlUoDCa7XIlLcnY{3HL%3G*&O|%{8JyrY1%Bc&K*eqAb$kU<TfbfJxAq`0Rv5WK)2Z5#E`ct_(27zQrj%fUwK3L}?n#QA3fswR-|Fv2!Xjji*C&{uPKVk7CwRj42t6GzWU*deJd|+ANA+mXh=Hpz$he!SGaxRmZKuv$$Y`DQF6cp+l;BLV8arTX`)>HI=z@t5}TPNZ79mZaB%4`g$>AwBmAt=8(r#$#38wjX4XO!Qc14daZ?7Ucf-QCEl%pI${$f1O7_<2(M=Tq_%MAF4Etz!oT=qFT3eT?Qp!NYRshzuKgU$=c%9EH!pm!^_iN$6v@=0@)fHZ)Yo;;PyXLlWiR=|%+>AP^;#x3qkMo{7pfd#_^i;d9IERXrD&eq^c)c4PCPNQbM;QCQqK?{SU$6<ThwuB-dO#PBR$zw`1n`Z}WgxKi3Mlq!wbJbIW4y3g<ER-VJ>=S8G&cqWF^LfOn$8)4ntPLzB<0pkdE%LnUeP@vg$8<#u;g0uqdP0X1X&Kn2AwxptwJGFKNg)@MBfO6%}58cpKW8b|V4hNKKzkaDJ-wk%M2#>y6Fre_scjg7TWDrCZpBVY;3vBhesjmN;3RpwOVol_!&^Y8l>d$X9_+2Uac=qODcw>rtK+=K&`^<@q!=1x0M6~?r*K9I|Q<?BEE)LZ+dbWwE!H2=so@sZ)a6VVeuW_ry@OLlMa~`7J0YFb`vH<59?tE(sQZwnIQh_#Oa!4GJ?1b5c)fQM3AXX7UUVe>DVEjncTF>(g5cGMKR;!BL7teWd0RjVHVc**xb0!rs4vf7ByFLIf3Y2O7Y555Vd8+hhweYcc{>yKTGtkzM7o&YXgFviFCY^5B2PD;RMD%#zbsF;RI1!2%-Z}b8FuMzr>qV8KQc*$m_^jkQ21wEIGVqG8g>zd?r=mqDP+LaTo_T5n+Qhdcd0gQ@*0ZvEq+|a;MWdTB2aIt(mfhf}kMk8``MJ{<(eq*7w`%KymX}?w`6Tzj2mZ~oA|^w)osn*|2f8!eamdA<g2@qu5sLRw=Uw^=Gt4RcUMcZz^G64zlMURx*^sr4{#p4X3$Q9un~utK1Ge$mMMtwP%r1aT&?O-Zx$|vpo?{?Lf*thjIT!Gv_Pk3@VS_mC?Uj=0FPPm*{QHtnWZW_2EE+llg=akr+YNeQW2e`m`sfgp`l_SK^r3+$xKlO%)(Gt5Z4s>uoB-Y@3g)g?PeL&p@1a1QQMkCh;K#}(d_J=(I}~3ca*i%Z&JrDP(R@m$F>ws!c`GX<d|-nhSM$%G5+^Y`wM)D&uh4Pr^%;WmBd~n^w}g(lK|l-^DJK6M1bxj>7d?)1KnN*bW;u-m6lbnk2*_Z3WhnZ<MVt*+%SdFiXuPi8$4rI7QJV$D_f}^{f$Y?ia~%#;KwDqqS+o(q7iS6T-X17vF1UMDhXX9>k)^pEEa+2h7TT-HgPv|{2`0i^%x<B4Dl!RKG7I}xF_I0ERg-R?JII579QI$7K{&vpS2FB~8Wo0^Y3!bv836U*cZ;g0Q*bGb^;Sk>637);P5-OUg56KFE!jjmR#!o{K5{C0dVj=P-O*n7;`aI!!|4G`Ki8|TosDn{sul!B*if3@kh#v454B51wPlmp@OhW|a-!oP+_jOh!N7<E<DP_;zW3{gjql$b5j)O;pQ?N{6J5A~(a=c0Hbeyrwd%Lyif4i8{!@P9dVCNw@+DDIq#h_=6@9PR+6df9ZINPPqp+%I>v*`+C}=G8i=D5T1XfbEM=BCWK*-C^zvj-2KsAkxp}Xy>A&G2jX0AvB(neF;r|)rrp(Mz8pvi@Hp$BvB{mp`vvfTI|4HHoEeoj!2+89KHg`#sRxNuU@{%lh82n_Bzea2p40zSAn{5dg`3)F~WZq8PGI4OfT={5Jmvd7Ow&S{Ro@nB^Y?+pWBdbJKL$TEQhzarl<e*~)!G2_SmJIKQGeSGh=Z9uYf#K|_92kewPoy2q*pnURg0mBv=45&(ewpIt9<2vS7uL>kY>Z`;mzXg+72CrRWQI~0f756<{&}z}?v(1YIrxi@Y!>Ck9K>Bun+0h5}?;XlU(g%P=<>?RA0=-z?VBtr<#UUkx3s1@R7+({T54N`^V|*jIx1%`@neO;`Lb`kyByI06TzrO~bMN)1ZkZsB_gt=ixyuC3t&czEw_@+{gcH)|Fdlf>AX0y!w+f3BBb0bM6=hHA*t_RX0KE%`l}vu(=Z&$3ZxD`J-7@*m<=+bzRJM-p`)wGA5sSlO7WvRJ!N^kh?I>pV!aDS-14<f_M~-+>LB=t@?B<1DuvD|LcwC|ju$%8W?|L(g^JD)T#bAtI_-VvuH*_L|qPdUQ4OS{o-ag&RhG9u{6J%)?G#rGqZA-KGdq&mV<Qbax!|JVIArB-M><au#j|nrH#2iGss30;ydri}5JIJA%Rf~RO1H<&%l-P-3SR(JM?e)DMl&BqWOq`;@&M^j2*N6hE_=@6ac0b5EM_$J;Wr6w~E=={ey&&Sq)1)5}gV19>Bv<kW7c`5X5wsTNK|6Peg_$iB7!hvRh753^*K5JRxsx1l7X6^<N$H2v&K=}G&1hihxc`UZhZGQ_;NZ~E#s*ouW17<YIbbC__5+Pdhcz45^;f;=1Z=Mu-I!Slh^n2RnYX4u#Ul^Pes<&U0H*xJQXG0pOz86KvmDT9`j^>I5zea_Zo5Mwv3j*5VoOno{APoqrMeyvycBfDHxuU-(ylKiepvroOF`6iG&eHZnH9x`{ER4L``#fasGdl=TaVvgnxTguCn2Xzn)f74Qy|sB@7veQ_<NBR{=hl~^`TUUZMVbjUy7;2%y%}t_@ma!@7n<QA5cpH0xbhA000080000X04o~a^h*E$0KfnM02=@R00000000000Du7i0001TVQy<*b8&EAV{dMBa&K&3b7Ns_WpgfWaCuNm0Rk-pEdT%j2mk;8AplRcCeh#x004jw000>P0000000000004jim;e9(a$#<3UvqJAUt@1>baHQOUq~`vNiJ@1c~DCM0xbhA000080000X08|_-KzaZG0KfnM02lxO00000000000Du9*4gdgUaA{|9VQ^?)a$#<3Uvpz&Y-MvUZg6=}O928c11$gm00;m803iS~r^UsF4gdgv4*&ob00000000000001h0f!F&0A+A#XL4b1XkT(+ZfjpiGG9q9Zg6=}O9ci10000400aOU0RRA19smFU00')

assert hashlib.sha256(FROZEN_BYTES).hexdigest()==FROZEN_MANIFEST['payload_sha256']
with np.load(io.BytesIO(FROZEN_BYTES),allow_pickle=False) as z:
    FROZEN={name:dict(scales=z[name+'_scales'].copy(),H2_I=z[name+'_H2_I'].copy()) for name in FROZEN_MANIFEST['arrays']}
for name,v in FROZEN.items():
    assert v['scales'].shape==(8,) and np.all(v['scales']>0) and v['H2_I'].shape==(8,60)
    assert hashlib.sha256(v['H2_I'].tobytes()).hexdigest()==FROZEN_MANIFEST['arrays'][name]['H2_I_sha256']
display(pd.DataFrame([dict(arm=a,wall_seconds=r['wall_seconds'],cpu_seconds=r['cpu_seconds'],maximum_wall_seconds=60,maximum_cpu_seconds=120,
    termination=r.get('status',r.get('termination')),lambda_eff=r['metrics']['lambda_eff']) for a,r in FROZEN_MANIFEST['historical']['arms'].items()]))


## 4. Restricted QROM domain and approximate DQI

The fresh loader audit covers all 1,220 addresses and four target patterns satisfying **bit 7 equals bit 6**, forward/inverse and phase before official reset/restoration. Borrowed slots are discovered from the actual circuits. A live invalid-target counterexample exposes the limit. This does not certify arbitrary dirty-target XOR. No circuit or verifier is patched.

The credited 889-qubit DQI is approximate. Every new salt reports value/phase sample errors, effective error and confidence/repetition penalties. Zero observed errors or PASS on these two salts does not guarantee other salts.


In [ ]:
"""Audit the clean sign-copy subspace used by the pinned angle loaders.

This does not certify a general dirty-target XOR oracle. The negative full-byte
fixture is retained: the optimized V12 loaders borrow public high bits as clean
workspace and are only tested/released on the invariant used by actual words.
"""
def validate_sign_copy_words(values):
    values=np.asarray(values,dtype=np.int64)
    if np.any(values<0) or np.any(values>255):
        raise ValueError('Expected 8-bit angle words.')
    if np.any(((values>>7)^(values>>6))&1):
        raise ValueError('QROM precondition failed: bit7 must equal bit6 in every word.')
    return values


def audit_sign_copy_qrom(km,circuits,angle_words,seed=281330):
    from dfthcsos.scoreinfra.circuits import register_layout,_check_state
    circuits=tuple(circuits);layout=register_layout(circuits)
    assert layout.brot==8 and layout.batch_size==5 and len(circuits)==15
    words=validate_sign_copy_words(angle_words).reshape(1220,15,5)
    deltas=words.copy();deltas[:,1:]^=words[:,:-1]
    validate_sign_copy_words(deltas)
    negative_rejected=False
    try:validate_sign_copy_words(np.array([[128,0,0,0,0]]))
    except ValueError:negative_rejected=True
    assert negative_rejected
    address=np.tile(np.arange(1220,dtype=np.int64),4)
    abits=((address[None,:]>>np.arange(len(layout.selection))[:,None])&1).astype(bool)
    place=np.int64(1)<<np.arange(8);rows=[];certificates=[]
    high={18,26,34,42,50}
    expected_cx=[f'CX q{17+8*j} q{18+8*j}' for j in range(5)]
    for k,c in enumerate(circuits):
        lines=str(c).splitlines()
        body=[s for s in lines if not s.startswith(('REGISTER ','APPEND_TO_REGISTER '))]
        measured_public=[]
        for line in body:
            parts=line.split()
            if parts[0] in ('HMR','R') and parts[1].startswith('q'):
                qid=int(parts[1][1:])
                if qid in layout.public_qubits:measured_public.append(qid)
        if measured_public:
            assert set(measured_public)==high
            assert body[:5]==expected_cx and body[-5:]==expected_cx
        certificates.append(dict(k=k,borrowed_sign_difference_qubits=sorted(set(measured_public)),prefix_suffix_certificate=bool(measured_public),public_register_measurement_count=len(measured_public)))
        low=np.random.default_rng(seed+k).integers(0,128,size=(len(address),5),dtype=np.int64)
        z=(low & 127)|((low & 64)<<1)
        z[:1220]=0;z[1220:2440]=255;validate_sign_copy_words(z)
        sim=km.Simulator(batch_size=len(address),debug_print=False,seed=seed+100+k)
        sim.use_same_registers_as(circuits[0])
        def restore(values):
            # Exactly the organizer protocol: check before clearing, then restore
            # public data so no measured classical state leaks into the next call.
            sim.clear_for_shot();sim.write_across_shots(layout.selection,abits)
            for j,word in enumerate(layout.words):
                sim.write_across_shots(word,((values[:,j][None,:]>>np.arange(8)[:,None])&1).astype(bool))
        restore(z)
        for label,want in [('forward',z^np.tile(deltas[:,k],(4,1))),('inverse',z)]:
            sim.do(c);_check_state(sim,c,layout,abits,f'legal-target {label} ANGLE_{k}')
            actual=np.stack([place@sim.read_across_shots(word).astype(np.int64) for word in layout.words],axis=1)
            assert np.array_equal(actual,want),(k,label)
            validate_sign_copy_words(actual)
            restore(actual)
        rows.append(dict(k=k,addresses=1220,target_patterns_per_address=4,forward_inverse_pass=True,phase_errors=0,value_errors=0,target_precondition='bit7 == bit6 in each word'))
    assert any(r['borrowed_sign_difference_qubits'] for r in certificates)
    # Keep a live falsification fixture. These are invalid compressed inputs,
    # and the failure must remain visible instead of being counted as a PASS.
    invalid=np.zeros((1220,5),dtype=np.int64);invalid[:,0]=128
    sim=km.Simulator(batch_size=1220,debug_print=False,seed=seed+900)
    sim.use_same_registers_as(circuits[0]);sim.clear_for_shot()
    aa=np.arange(1220,dtype=np.int64)
    bits=((aa[None,:]>>np.arange(len(layout.selection))[:,None])&1).astype(bool)
    sim.write_across_shots(layout.selection,bits)
    for j,word in enumerate(layout.words):
        sim.write_across_shots(word,((invalid[:,j][None,:]>>np.arange(8)[:,None])&1).astype(bool))
    sim.do(circuits[0])
    got=np.stack([place@sim.read_across_shots(word).astype(np.int64) for word in layout.words],axis=1)
    bad=np.any(got!=(invalid^deltas[:,0]),axis=1)
    counterexample=dict(k=0,initial_target=[128,0,0,0,0],addresses=1220,value_errors=int(np.count_nonzero(bad)),phase_errors=int(np.count_nonzero(sim.read_phase_flipped_across_shots())),expected_to_fail=True)
    assert counterexample['value_errors']==1220
    return dict(status='PASS_ON_SIGN_COPY_SUBSPACE',universal_dirty_target_xor_certified=False,all_91500_words_and_deltas_meet_precondition=True,negative_invalid_target_rejected_before_simulation=negative_rejected,out_of_subspace_counterexample=counterexample,certificates=certificates,circuits=rows,organizer_reset_protocol=True)


## 5. New full official verification

Attach the official competition and **PUBLIC Free-Gap SQP V1 363660552 output**, ZIP SHA196eaa849e3a28319d7df5bc1308993d23e9adfd35babdec4d4e8c1d47ea5335. Use the Python3.13 CPU image, internet off and two threads. Source, smooth rank8 SQP control and epigraph endpoint each receive 78,080 chemistry shots per salt. The unchanged invsqrt receives 200,540,928 shots and DQI 102,400. Salts: `avazaku-epigraph-rank-a-20261030` and `avazaku-epigraph-rank-b-20261030`.

All ten reports must be fresh, full and unmodified. The candidate must be cheaper on both salts, T 16,266 / Q 430. Frozen factor hashes, official numerical parity, fixed one-body target, errors below0.02, positive gap, CRC and all 18 archive members are independent gates. Only `W`, `H2_I` and derived `offset_W = H2_I - W.sum(axis=1)` may change within the factor NPZ. All 17 circuits and other arrays must remain byte-identical.


In [ ]:
ROOT=Path('/kaggle/working');OUT=ROOT/'epigraph_rank_results';OUT.mkdir(exist_ok=True)
START=time.monotonic();SALTS=['avazaku-epigraph-rank-a-20261030','avazaku-epigraph-rank-b-20261030']
def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def write(name,obj):
    p=OUT/name;t=p.with_suffix(p.suffix+'.tmp');t.write_text(json.dumps(obj,indent=2,default=lambda v:v.item() if isinstance(v,np.generic) else str(v)));t.replace(p)
def event(name,**kw):
    row=dict(event=name,elapsed_seconds=time.monotonic()-START,**kw);write('progress.json',row)
    with (OUT/'events.jsonl').open('a') as f:f.write(json.dumps(row)+'\n')
    print(json.dumps(row),flush=True)
def main():
    inputs=Path('/kaggle/input');comp=next((p.parents[1] for p in inputs.rglob('kickmix.so') if p.parent.name=='kickmix' and all((p.parents[1]/n).is_file() and sha(p.parents[1]/n)==h for n,h in COMPETITION_FILES.items())),None)
    assert comp is not None,'Attach unmodified official competition inputs.'
    assert sys.version_info[:2]==(3,13),'Use Python3.13 CPU image.'
    sys.path[:0]=[str(comp/'kickmix'),str(comp/'verifiers')]
    import kickmix as km
    from dfthcsos import verify as chem
    from dfthcsos.scoreinfra import reference,dfthc_math as maths
    from invsqrt import verify_submission_invsqrt as inv
    from dqi import verify_submission_dqi as dqi
    archive=next((p for p in inputs.rglob('submission.zip') if sha(p)==ARCHIVE_SHA),None)
    assert archive is not None,'Attach exact PUBLIC Free-Gap SQP V1 363660552 output, ZIP196eaa...'
    Path('/kaggle/temp').mkdir(exist_ok=True);temp=Path(tempfile.mkdtemp(prefix='epigraph-rank-',dir='/kaggle/temp'));base=temp/'source';base.mkdir()
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None and len(z.namelist())==18 and set(z.namelist())==set(SOURCE_MEMBERS)
        for name in z.namelist():
            p=PurePosixPath(name);assert not p.is_absolute() and '..' not in p.parts
            target=base/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name));assert sha(target)==SOURCE_MEMBERS[name]
    assert sha(base/'dfthcsos/submission.npz')==FACTOR_SHA
    with np.load(base/'dfthcsos/submission.npz',allow_pickle=False) as z:data={k:np.array(z[k]) for k in z.files}
    w,h=data['W'],data['H2_I'];assert w.shape==(8,143,60)
    write('source_manifest.json',dict(version=363660552,source_public_score=22.88871,archive_sha256=ARCHIVE_SHA,factor_sha256=FACTOR_SHA,members=SOURCE_MEMBERS,original_portfolio_version=363642779,credits=['Heliosli/Dmitrii/Ahmed/sigmaborov/Baidalin weights; Atakan words on SomeJoint architecture','Alex Chilton/Kirill/Dmitrii/Heliosli/Atakan loader assembly','hengck23 approximate889DQI; mrmeows/busyaprime63Qinvsqrt; complete credited decoder lineage','Our prior Free-Gap SQP V1 19.921ppm chemistry study; extra gain here is separate'],own_change='Standard LP epigraph rank-amplitude method versus smooth rank8 SQP, one-body compensation; W/H2I/derivedoffset_W only.'))
    ref=reference.load_reference(reference.check_fcidump());fixed=reference.load_fixed_shifts();n=int(ref['orbitals']);shift=np.asarray(maths.symmetric_from_lower(fixed['Hsym_lower'],n),dtype=np.float64);beta=float(float(fixed['H1_I'])-.5*float(np.trace(shift)))
    def numerical(ww,hh):return maths.evaluate_dfthc_angle_words(data['angle_words'],ww,hh-ww.sum(axis=1),shift,beta,ref,brot=int(data['brot']),tree_edges=data['tree_edges'],energy_upper=reference.UPPER_ENERGY)
    source=numerical(w,h);assert abs(source['lambda_eff']-48.35914523441181)<1e-8
    vectors=maths.reconstruct_u_vectors(data['angle_words']/128,data['tree_edges']);vectors/=np.linalg.norm(vectors,axis=1,keepdims=True)
    body=vectors[n:].reshape(8,143,n);x=maths.reconstruct_x(body,w);x=np.tril(x)+np.swapaxes(np.tril(x,k=-1),-1,-2)
    base_h1=ref['oei']-.5*np.einsum('prrq->pq',ref['tei'])-.5*int(ref['electrons'])*shift+beta*np.eye(n);constant=float(reference.UPPER_ENERGY)-float(ref['ecore'])+int(ref['electrons'])*beta
    model=OffsetModel(w,x,vectors[:n],base_h1,constant,assignment=linear_sum_assignment);p0=h-w.sum(axis=1);prediction=model.evaluate(p0,False)
    parity={k:abs(source[k]-prediction[k]) for k in ['error_1','Lambda','E_gap','lambda_eff']};assert max(parity.values())<1e-8
    amp=RankAmplitudeModel(w,h,x,source,ref,shift,maths);controls=dict(source_model_parity=parity,rank_model=test_rank_model(amp,numerical));write('controls.json',controls)
    np.savez(OUT/'quadratic_error2_model.npz',A=amp.A,d=amp.d,e20=amp.e20);write('negative_screens.json',NEGATIVE_SCREENS);event('fresh_model_and_epigraph_controls_pass',metrics=source)
    arms={'source':dict(W=w,H2_I=h)}
    for name in FROZEN:
        ww=w*FROZEN[name]['scales'][:,None,None];hh=FROZEN[name]['H2_I'];pin=FROZEN_MANIFEST['arrays'][name]
        assert hashlib.sha256(ww.tobytes()).hexdigest()==pin['W_sha256'];assert hashlib.sha256(hh.tobytes()).hexdigest()==pin['H2_I_sha256'];arms[name]=dict(W=ww,H2_I=hh)
    metrics={name:numerical(v['W'],v['H2_I']) for name,v in arms.items()};defects={};source_matrix=base_h1-(p0.ravel()@x.reshape(480,-1)).reshape(n,n)
    for name,m in metrics.items():
        assert m['error_1']<.02 and m['error_2']<.02 and m['E_gap']>3.914898 and m['gap_in_domain']
        assert abs(m['error_1']-source['error_1'])<1e-10
        ww,hh=arms[name]['W'],arms[name]['H2_I'];xx=maths.reconstruct_x(body,ww);xx=np.tril(xx)+np.swapaxes(np.tril(xx,k=-1),-1,-2)
        offset=hh-ww.sum(axis=1);matrix=base_h1-(offset.ravel()@xx.reshape(480,-1)).reshape(n,n);defect=float(np.max(abs(matrix-source_matrix)));assert defect<1e-10
        pred=OffsetModel(ww,xx,vectors[:n],base_h1,constant,assignment=linear_sum_assignment).evaluate(offset,False);assert max(abs(pred[k]-m[k]) for k in ['error_1','Lambda','E_gap','lambda_eff'])<1e-8
        if name!='source':
            small=amp.evaluate(np.log(FROZEN[name]['scales']));assert max(abs(small[k]-m[k]) for k in ['error_2','Lambda','E_gap','lambda_eff'])<1e-8
            assert abs(m['lambda_eff']-FROZEN_MANIFEST['historical']['arms'][name]['metrics']['lambda_eff'])<1e-8
        defects[name]=defect
    winner='epigraph_rank';assert min(1e6*(1-metrics[winner]['lambda_eff']/metrics[n]['lambda_eff']) for n in ['source','rank_sqp_control'])>.01
    write('numerical.json',metrics);write('one_body_invariance.json',defects);write('frozen_manifest.json',FROZEN_MANIFEST)
    rows=[dict(arm=a,lambda_eff=m['lambda_eff'],error1_mHa=m['error_1']*1000,error2_mHa=m['error_2']*1000,gap=m['E_gap'],additional_gain_ppm=1e6*(1-m['lambda_eff']/source['lambda_eff'])) for a,m in metrics.items()];table=pd.DataFrame(rows);display(table);table.to_csv(OUT/'comparison.csv',index=False)
    folders={'source':base}
    for name in FROZEN:
        folder=temp/name;folder.mkdir()
        for member in SOURCE_MEMBERS:
            if member=='dfthcsos/submission.npz':continue
            dest=folder/member;dest.parent.mkdir(parents=True,exist_ok=True);os.link(base/member,dest)
        changed={k:np.array(v,copy=True) for k,v in data.items()};changed.update(W=arms[name]['W'].copy(),H2_I=arms[name]['H2_I'].copy());changed['offset_W']=changed['H2_I']-changed['W'].sum(axis=1)
        assert all(np.array_equal(changed[k],data[k]) for k in data if k not in ['W','H2_I','offset_W'])
        np.savez(folder/'dfthcsos/submission.npz',**changed);np.savez(OUT/(name+'_factor.npz'),**changed);assert sha(folder/'dfthcsos/submission.npz')==FROZEN_MANIFEST['arrays'][name]['factor_sha256'];folders[name]=folder
    circuits=[km.Circuit.from_file(str(base/f'dfthcsos/angle_k_{k}.kmb'),format='kmb') for k in range(15)];qrom=audit_sign_copy_qrom(km,circuits,data['angle_words']);write('qrom_sign_copy_audit.json',qrom);del circuits;gc.collect();event('restricted_qrom_pass',counterexample=qrom['out_of_subspace_counterexample'])
    def full(fn,*args,**kw):
        if 'max_workers' in inspect.signature(fn).parameters:kw['max_workers']=2
        return fn(*args,**kw)
    chemistry=[]
    for salt in SALTS:
        group=dict(salt=salt,reports={})
        for name,folder in folders.items():
            event('full_chemistry',arm=name,salt=salt);r=full(chem.verify,str(folder/'dfthcsos'),salt=salt)
            assert r['status']=='PASS' and r['verified_shots']==78080 and r['toffoli_count']==16266 and r['num_qubits']==430
            assert abs(r['lambda_eff']-metrics[name]['lambda_eff'])<1e-8;group['reports'][name]=r
        assert group['reports'][winner]['score']<min(r['score'] for name,r in group['reports'].items() if name!=winner)
        chemistry.append(group);write('chemistry_full.json',chemistry)
    portfolio=[]
    for salt in SALTS:
        event('full_invsqrt',salt=salt);c=km.Circuit.from_file(str(base/'invsqrt.kmb'),format='kmb');ir=full(inv.verify,c,inv.ProblemSpecInvSqrt.for_n_p(10),salt=salt);assert ir['status']=='PASS' and ir['verified_shots']==200540928;del c;gc.collect()
        event('full_dqi',salt=salt);c=km.Circuit.from_file(str(base/'dqi.kmb'),format='kmb');dr=full(dqi.verify,c,dqi.ProblemSpecDQI.benchmark_instance(),salt=salt);assert dr['status']=='PASS' and dr['verified_shots']==102400;del c;gc.collect();portfolio.append(dict(salt=salt,invsqrt=ir,dqi=dr));write('portfolio_full.json',portfolio)
    checks=[dict(salt=g['salt'],task='chemistry',arm=a,status=r['status'],shots=r['verified_shots'],cost=r['score']) for g in chemistry for a,r in g['reports'].items()]
    checks += [dict(salt=g['salt'],task=t,arm='fixed upstream circuit',status=g[t]['status'],shots=g[t]['verified_shots'],cost=g[t]['score'],value_errors=g[t].get('value_errors'),phase_errors=g[t].get('phase_errors'),effective_error_rate=g[t].get('effective_error_rate'),repetition_factor=g[t].get('repetition_factor')) for g in portfolio for t in ['invsqrt','dqi']]
    assert len(checks)==10;display(pd.DataFrame(checks));pd.DataFrame(checks).to_csv(OUT/'native_checks.csv',index=False)
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False});labels=['Scored Free-Gap V1','Smooth rank8 SQP','L1 epigraph LP']
    fig,ax=plt.subplots(2,1,figsize=(7.6,6.8),layout='constrained');ax[0].barh(labels,table['additional_gain_ppm'],color=['#88959c','#c99c50','#007e87']);ax[0].invert_yaxis();ax[0].set(xlabel='Additional chemistry normalization saving (ppm)',title='Only about 0.00000338% beyond the scored source')
    hist=FROZEN_MANIFEST['historical']['arms']['epigraph_rank']['history'];ax[1].step([0]+[r['iteration'] for r in hist],[0]+[1e6*(1-r['lambda_eff']/source['lambda_eff']) for r in hist],where='post',color='#007e87');ax[1].scatter([hist[-1]['iteration']],[float(table.iloc[-1]['additional_gain_ppm'])],marker='D',s=45,color='#007e87',label='Verified saved endpoint');ax[1].legend(fontsize=8);ax[1].set(xlabel='Recorded LP iteration',ylabel='Best feasible gain (ppm)',title='Exact nonlinear acceptance, including rejected steps');fig.savefig(OUT/'controlled_gains.png',dpi=160);plt.show();plt.close(fig)
    fig,ax=plt.subplots(2,1,figsize=(7.6,6.4),layout='constrained');ax[0].barh(labels,[(.02-m['error_2'])*1e6 for m in metrics.values()],color='#007e87');ax[0].invert_yaxis();ax[0].set(xlabel='Error 2 margin below 20 mHa (microhartree)',title='Two-body error changes and remains below the cap');ax[1].barh(labels,[(m['E_gap']-3.914898)*1e6 for m in metrics.values()],color='#576d9f');ax[1].invert_yaxis();ax[1].set(xlabel='Gap above official floor (microhartree)',title='One-body target fixed; energy gap changes');fig.savefig(OUT/'physical_margins.png',dpi=160);plt.show();plt.close(fig)
    release=ROOT/'submission.zip'
    with zipfile.ZipFile(release,'w',zipfile.ZIP_DEFLATED) as z:
        for name in sorted(SOURCE_MEMBERS):z.write(folders[winner]/name,name)
    with zipfile.ZipFile(release) as z:assert z.testzip() is None and set(z.namelist())==set(SOURCE_MEMBERS)
    members={name:sha(folders[winner]/name) for name in sorted(SOURCE_MEMBERS)};assert [name for name in members if members[name]!=SOURCE_MEMBERS[name]]==['dfthcsos/submission.npz']
    with np.load(folders[winner]/'dfthcsos/submission.npz',allow_pickle=False) as z:changed_keys=[k for k in data if not np.array_equal(z[k],data[k])]
    assert set(changed_keys)=={'W','H2_I','offset_W'}
    summary=dict(status='FULLY_VERIFIED_OWN_EPIGRAPH_RANK_STUDY',winner=winner,source_version=363660552,source_public_score=22.88871,competition_public_score=None,submission_sent=False,fresh_full_receipts=10,salts=SALTS,numerical=metrics,controls=controls,one_body_invariance=defects,body_Gram_and_error2_fixed=False,whole_Hamiltonian_invariant=False,chemistry=chemistry,portfolio=portfolio,qrom_audit=qrom,archive_sha256=sha(release),candidate_factor_sha256=members['dfthcsos/submission.npz'],members=members,changed_members=['dfthcsos/submission.npz'],changed_npz_keys=changed_keys,coefficient_search_repeated=False,own_additional_gain_vs_source_ppm=1e6*(1-metrics[winner]['lambda_eff']/source['lambda_eff']),own_gain_vs_sqp_control_ppm=1e6*(1-metrics[winner]['lambda_eff']/metrics['rank_sqp_control']['lambda_eff']),historical_optimization=FROZEN_MANIFEST['historical'],elapsed_seconds=time.monotonic()-START)
    write('summary.json',summary);write('release.json',summary);event('complete',status=summary['status'],archive_sha256=summary['archive_sha256']);display(FileLink('submission.zip'))
    print('All10 NEW full checks passed. This is not a measured new competition score; such a small gain may leave the5-decimal display unchanged.')
try:main()
except Exception as exc:
    import traceback
    event('FAILED',error=repr(exc),traceback=traceback.format_exc());raise


## 6. Limits, source and attribution

The additional gain is tiny: about 0.00000338% chemistry normalization, not a promised measurable improvement in the displayed competition score. The LP result depends on this source and local feasible region. Approximate KKT diagnostics, failed SQP, finite trust-region iterations and two salts do not prove global optimality or all-salt success. W/bodyGram/error2 change deliberately; only the one-body target is preserved. Numerical estimates, native receipts and an actual new competition submission score remain different quantities.

The immediate source is [our Free-Gap SQP V1](https://www.kaggle.com/code/avazaku/free-gap-sqp-889-qubit-dqi?scriptVersionId=363660552), based on [Heliosli V21](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy?scriptVersionId=363642779). Source weights and portfolio: Heliosli, [Dmitrii V38](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=363646243), Ahmed Berat Ozer, sigmaborov and Adilzhan Baidalin. Angle words: Atakan Aldemir on SomeJoint's R8/B143/C60 architecture. Loader assembly: Alex Chilton, Dmitrii, Heliosli, Atakan and Kirill. Approximate 889 DQI: hengck23 on SomeJoint/Agniv Banerjee, Ahmed, Heliosli, Alex Chilton, jday96314, manwithacat and the credited decoder lineage. Invsqrt: mrmeows and busyaprime. Full historical lineage remains in the pinned upstream notebooks. We do not claim those circuits, architectures or starting weights.

For reuse, compare the explicit epigraph with your own same-start smooth control, keep the exact nonlinear acceptance rule and negative records, and repeat full native checks after every coefficient or circuit change.
